In [ ]:
def plot_blocks_and_singular_values(P_blocks, Q_blocks,
                                   num_blocks, current_rank,
                                   out_dir, max_blocks=6):
    """
    2-row figure:
      Row 1: core matrices M_bb = Q_b^T P_b (rank x rank)
      Row 2: singular values of each M_bb

    Big layout + large fonts for readability.
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    os.makedirs(out_dir, exist_ok=True)

    n_plot = min(max_blocks, num_blocks)
    if n_plot <= 0 or current_rank <= 0:
        print(f"Nothing to plot: n_plot={n_plot}, rank={current_rank}")
        return

    # ---- global style ----
    plt.rcParams.update({
        "font.size": 22,
        "axes.titlesize": 26,
        "axes.labelsize": 22,
        "xtick.labelsize": 20,
        "ytick.labelsize": 20,
        "legend.fontsize": 20,
    })

    # Big canvas: width grows with number of blocks
    W = 8.5 * n_plot   # 7 inches per block column
    H = 14.0           # total height
    fig, axes = plt.subplots(2, n_plot, figsize=(W, H))

    if n_plot == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    # ---- Row 1: core matrices ----
    for i in range(n_plot):
        b = i
        M = (Q_blocks[b].T @ P_blocks[b]).detach().cpu().numpy()

        ax = axes[0, i]
        sns.heatmap(
            M,
            annot=True if current_rank <= 8 else False,
            fmt=".2f",
            cmap="vlag",
            square=True,
            ax=ax,
            cbar=(i == n_plot - 1),
            cbar_kws={"shrink": 0.8}
        )
        ax.set_title(f"Block {b+1}: core $Q^T P$ (r={current_rank})")
        ax.set_xlabel("rank comp.")
        ax.set_ylabel("rank comp.")

    # ---- Row 2: singular values ----
    for i in range(n_plot):
        b = i
        M = (Q_blocks[b].T @ P_blocks[b]).detach().cpu().numpy()
        s = np.linalg.svd(M, compute_uv=False)

        ax = axes[1, i]
        ax.plot(np.arange(1, len(s) + 1), s, marker="o", linewidth=3)
        ax.set_title(f"Block {b+1}: singular values")
        ax.set_xlabel("Index")
        ax.set_ylabel("Singular value")
        ax.grid(True, linestyle="--", alpha=0.4)
        ax.set_xticks(np.arange(1, len(s) + 1))

    plt.tight_layout()
    out_path = os.path.join(out_dir, f"blocks_core_and_svals_rank{current_rank}_big.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_path)


In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# os.environ["TORCH_USE_CUDA_DSA"] = "1"  # optional, requires restart

import torch
import numpy as np
from torch.utils.data import DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import tiktoken

# from your AdaGramFR implementation
# from adagram_optimizers.AdagramSVD import AdaGramFR
# from adagram_optimizers.AdagramPS import AdaGramFR
from adagram_optimizers.AdamGram import AdamGram

# from nanoGPT repo
from model import GPT, GPTConfig


def make_xy_from_text(text: str, vocab_size: int, block_size: int, device, enc):
    """
    nanoGPT-style: encode text -> clamp to vocab -> truncate to block_size+1 -> shift to x,y.
    Same logic as your make_batch_from_text(). 
    """
    ids = enc.encode(text)
    ids = [min(max(int(t), 0), vocab_size - 1) for t in ids]
    ids = ids[: block_size + 1]  # +1 for shift

    if len(ids) < 2:
        # ensure at least one token in x and y
        ids = ids + [0]

    x = torch.tensor(ids[:-1], dtype=torch.long, device=device)[None, :]  # (1, T)
    y = torch.tensor(ids[1:],  dtype=torch.long, device=device)[None, :]  # (1, T)
    return x, y



device = torch.device("cpu")
print(f"Using device: {device}")
# ----------------- load nanoGPT checkpoint -----------------
# ckpt_path = "out-sgd/5000_ckpt.pt"
# ckpt_path = "out_muonwithadamw_bs16_lr1e-3/5000_ckpt.pt"
# ckpt_path = "out_adamw_bs16_lr1e-4/5000_ckpt.pt"
ckpt_path = "out_adagramps_bs512_lr1e-2_rank1/5000_ckpt.pt"



print(f"Loading nanoGPT checkpoint from {ckpt_path}...")
checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
# Build model from model_args (nanoGPT style)
config = GPTConfig(**checkpoint["model_args"])
if hasattr(config, "flash"):
    config.flash = False
model = GPT(config)
# Strip _orig_mod. prefix (torch.compile artifact)
sd = checkpoint["model"] if "model" in checkpoint else checkpoint
sd = {k.replace("_orig_mod.", ""): v for k, v in sd.items()}
# strict=False ignores extra keys like attn.bias
model.load_state_dict(sd, strict=False)
model.to(device)
model.eval()
# ----------------- tokenizer / encoding (nanoGPT-style) -----------------
enc = tiktoken.get_encoding("gpt2")  # same as nanoGPT prepare.py uses [web:112]
vocab_size = int(checkpoint["model_args"]["vocab_size"])
model_block_size = int(checkpoint["model_args"]["block_size"])
block_size = min(128, model_block_size)
print(f"Checkpoint vocab_size={vocab_size}, block_size={model_block_size}, using T={block_size}")
# ----------------- select transformer blocks -----------------
num_layers_to_analyze = 6  # your setting
transformer_blocks = model.transformer.h[:num_layers_to_analyze]
num_blocks = len(transformer_blocks)
print(f"Analyzing first {num_blocks} transformer blocks.")
# ----------------- collect param sizes per block -----------------
block_param_sizes = []
total_params = 0
for block in transformer_blocks:
    block_size_params = sum(p.numel() for p in block.parameters() if p.requires_grad)
    block_param_sizes.append(block_size_params)
    total_params += block_size_params
print(f"Total parameters in first {num_blocks} blocks: {total_params:,}")
print("Block sizes:", block_param_sizes)
# ----------------- dataset + dataloader -----------------
print("Loading dataset...")
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train[:50]")
# Keep raw text; we'll encode with tiktoken ourselves
dataset = dataset.filter(lambda x: isinstance(x["text"], str) and len(x["text"].strip()) > 0)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
# ----------------- fake param + AdaGramFR -----------------
print("Initializing AdaGramFR optimizer for first blocks...")
fake_param = torch.nn.Parameter(
    torch.zeros(total_params, device=device, dtype=torch.float32),
    requires_grad=True,
)
ada_gram_fr = AdamGram(
    [fake_param],
    lr=1e-5,
    eps=1e-4,
    weight_decay=0,
    max_rank=5,
    task="FirstBlocksAnalysis",
    enable_logging=False,
    save_matrix=False,
)
# ----------------- accumulate Fisher-style info -----------------
print("Processing batches to accumulate Fisher information...")
total_examples = min(50, len(dataset))
for batch_idx, batch in enumerate(tqdm(dataloader, desc="Processing batches", total=total_examples)):
    if batch_idx >= total_examples:
        break
    text = batch["text"][0]  # batch_size=1
    x, y = make_xy_from_text(text, vocab_size=vocab_size, block_size=block_size, device=device, enc=enc)
    model.zero_grad(set_to_none=True)
    # Depending on your nanoGPT model.py, ONE of these will work:
    # 1) karpathy nanoGPT: logits, loss = model(x, y)
    # logits, loss = model(x, y)
    # 2) your variant (you previously used targets=)
    logits, loss = model(x, targets=y)
    loss.backward()
    all_grads = []
    for block in transformer_blocks:
        for p in block.parameters():
            if p.grad is not None:
                all_grads.append(p.grad.detach().flatten().to(torch.float32))
    if all_grads:
        full_grad_vector = torch.cat(all_grads)
        if full_grad_vector.numel() != total_params:
            raise ValueError(f"Gradient length {full_grad_vector.numel()} != total_params {total_params}")
        fake_param.grad = full_grad_vector
        ada_gram_fr.step()
    model.zero_grad(set_to_none=True)
    fake_param.grad = None
# ----------------- extract P, Q -----------------
print("Extracting P and Q matrices from AdaGramFR...")
state = ada_gram_fr.state[fake_param]
if "P" in state and "Q" in state:
    P_full = state["P"].clone()
    Q_full = state["Q"].clone()
    current_rank = P_full.shape[1] if P_full.dim() > 1 else 1
    print(f"Full P shape: {P_full.shape}, Full Q shape: {Q_full.shape}")
    print(f"Current rank: {current_rank}")
else:
    print("Warning: P and Q matrices not found in optimizer state.")
    P_full = torch.zeros((total_params, 2), device=device)
    Q_full = torch.zeros((total_params, 2), device=device)
    current_rank = 0
# ----------------- split P, Q by blocks -----------------

print("Splitting P and Q into blocks...")
P_blocks, Q_blocks = [], []
start_idx = 0
for block_idx, bs in enumerate(block_param_sizes):
    end_idx = start_idx + bs
    P_blocks.append(P_full[start_idx:end_idx, :])
    Q_blocks.append(Q_full[start_idx:end_idx, :])
    print(f"Block {block_idx+1}: size={bs}, P={P_blocks[-1].shape}, Q={Q_blocks[-1].shape}")
    start_idx = end_idx
# ----------------- pairwise norms -----------------
print("Computing pairwise block spectral norms...")
block_norms = torch.zeros((num_blocks, num_blocks), device=device)
for i in tqdm(range(num_blocks), desc="Computing norms"):
    P_i = P_blocks[i]
    for j in range(num_blocks):
        Q_j = Q_blocks[j]
        if P_i.shape[0] > 0 and Q_j.shape[0] > 0 and current_rank > 0:
            small_matrix = Q_j.T @ P_i  # (rank, rank)
            block_norms[i, j] = torch.linalg.norm(small_matrix).item()  # Frobenius of rankxrank
block_norms_np = block_norms.cpu().numpy()
out_dir = "results_nanogpt_blocks"


In [ ]:
# =========================
# PLOTTING SCRIPT (paste after block_norms_np is computed)
# =========================
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm import tqdm

PLOT_DIR = f"plots_{SPLIT_MODE}_250"
os.makedirs(PLOT_DIR, exist_ok=True)

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print(f"[saved] {path}")
    plt.close()

# ---------- 1) BLOCK x BLOCK heatmap ----------
plt.figure(figsize=(7, 6))
ax = sns.heatmap(
    block_norms_np,
    annot=True if num_blocks <= 12 else False,
    fmt=".2e",
    square=True,
    cbar=True,
    xticklabels=[f"B{i}" for i in range(num_blocks)],
    yticklabels=[f"B{i}" for i in range(num_blocks)],
)
ax.set_title(f"Block-to-Block Frobenius norms || (P_i Q_j^T) ||_F  (rank={current_rank})")
ax.set_xlabel("Block j (Q slice)")
ax.set_ylabel("Block i (P slice)")
savefig(os.path.join(PLOT_DIR, "block_block_norms.png"))

# ---------- 2) BLOCK x BLOCK heatmap (log10) ----------
# Add epsilon so zeros don't become -inf
eps = 1e-30
log_block = np.log10(block_norms_np + eps)

plt.figure(figsize=(7, 6))
ax = sns.heatmap(
    log_block,
    annot=True if num_blocks <= 12 else False,
    fmt=".2f",
    square=True,
    cbar=True,
    xticklabels=[f"B{i}" for i in range(num_blocks)],
    yticklabels=[f"B{i}" for i in range(num_blocks)],
)
ax.set_title(f"log10 Block-to-Block norms (rank={current_rank})")
ax.set_xlabel("Block j (Q slice)")
ax.set_ylabel("Block i (P slice)")
savefig(os.path.join(PLOT_DIR, "block_block_norms_log10.png"))

# ---------- 3) BLOCK summaries (row/col norms) ----------
# Row sum: how strongly block i (P) interacts with all Q blocks
row_sum = block_norms_np.sum(axis=1)
col_sum = block_norms_np.sum(axis=0)

plt.figure(figsize=(8, 4))
plt.plot(range(num_blocks), row_sum, marker="o", label="Row-sum (fixed P_i, sum over Q_j)")
plt.plot(range(num_blocks), col_sum, marker="s", label="Col-sum (fixed Q_j, sum over P_i)")
plt.xticks(range(num_blocks), [f"B{i}" for i in range(num_blocks)])
plt.yscale("log")  # usually useful
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.title("Block interaction strength (log scale)")
plt.xlabel("Block index")
plt.ylabel("Sum of norms")
plt.legend()
savefig(os.path.join(PLOT_DIR, "block_summaries.png"))

# ---------- 4) OPTIONAL: PART x PART heatmap ----------
# If you want it, compute part_norms and plot it.
DO_PART_PLOT = True

if DO_PART_PLOT:
    # Deterministic order of parts using part_meta (already in block order)
    part_keys = [(m["block"], m["part"]) for m in part_meta]
    part_labels = [f"B{b}:{p}" for (b, p) in part_keys]
    K = len(part_keys)

    part_norms = torch.zeros((K, K), device=device)

    for i in tqdm(range(K), desc="Computing part norms"):
        Pi = P_parts[part_keys[i]]  # (n_i, r)
        if Pi.numel() == 0:
            continue
        for j in range(K):
            Qj = Q_parts[part_keys[j]]  # (n_j, r)
            if Qj.numel() == 0:
                continue
            part_norms[i, j] = frob_norm_lowrank(Pi, Qj)

    part_norms_np = part_norms.detach().cpu().numpy()

    # (a) linear heatmap
    plt.figure(figsize=(max(10, K * 0.35), max(8, K * 0.35)))
    ax = sns.heatmap(
        part_norms_np,
        square=True,
        cbar=True,
        xticklabels=part_labels,
        yticklabels=part_labels,
    )
    ax.set_title(f"Part-to-Part Frobenius norms || (P_part Q_part^T) ||_F  (rank={current_rank})")
    ax.set_xlabel("Part j (Q)")
    ax.set_ylabel("Part i (P)")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    savefig(os.path.join(PLOT_DIR, "part_part_norms.png"))

    # (b) log heatmap
    log_part = np.log10(part_norms_np + eps)
    plt.figure(figsize=(max(10, K * 0.35), max(8, K * 0.35)))
    ax = sns.heatmap(
        log_part,
        square=True,
        cbar=True,
        xticklabels=part_labels,
        yticklabels=part_labels,
    )
    ax.set_title(f"log10 Part-to-Part norms (rank={current_rank})")
    ax.set_xlabel("Part j (Q)")
    ax.set_ylabel("Part i (P)")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    savefig(os.path.join(PLOT_DIR, "part_part_norms_log10.png"))

    # Save arrays too (handy for later)
    np.save(os.path.join(PLOT_DIR, "block_norms.npy"), block_norms_np)
    np.save(os.path.join(PLOT_DIR, "part_norms.npy"), part_norms_np)
    print(f"[saved] {os.path.join(PLOT_DIR, 'block_norms.npy')}")
    print(f"[saved] {os.path.join(PLOT_DIR, 'part_norms.npy')}")

In [ ]:
import os
import numpy as np
import torch
from tqdm import tqdm
import tiktoken
from datasets import load_dataset
from torch.utils.data import DataLoader

from adagram_optimizers.AdamGram import AdamGram
from model import GPT, GPTConfig

# ── config ────────────────────────────────────────────────────────────────────
CKPT_DIR       = "out_adamw_bs16_lr1e-4"
CKPT_STEPS     = list(range(250, 5001, 250))
SPLIT_MODE     = "fine"
NPY_DIR        = f"adamw_npys_{SPLIT_MODE}_all_ckpts"
PLOT_DIR       = f"adamw_{SPLIT_MODE}_all_ckpts"
NUM_LAYERS     = 6
NUM_EXAMPLES   = 50
BLOCK_SIZE_CAP = 128
MAX_RANK       = 5
STEP_WIDTH     = len(str(max(CKPT_STEPS)))

os.makedirs(NPY_DIR, exist_ok=True)
device = torch.device("cpu")

# ── helpers ───────────────────────────────────────────────────────────────────
def make_xy_from_text(text, vocab_size, block_size, device, enc):
    ids = enc.encode(text)
    ids = [min(max(int(t), 0), vocab_size - 1) for t in ids]
    ids = ids[: block_size + 1]
    if len(ids) < 2:
        ids = ids + [0]
    x = torch.tensor(ids[:-1], dtype=torch.long, device=device)[None, :]
    y = torch.tensor(ids[1:],  dtype=torch.long, device=device)[None, :]
    return x, y

def frob_norm_lowrank(U, V):
    GU = U.T @ U
    GV = V.T @ V
    return torch.sqrt(torch.trace(GV @ GU).clamp_min(0))

def iter_parts(block):
    return [
        ("ln_1",        block.ln_1),
        ("attn.c_attn", block.attn.c_attn),
        ("attn.c_proj", block.attn.c_proj),
        ("ln_2",        block.ln_2),
        ("mlp.c_fc",    block.mlp.c_fc),
        ("mlp.c_proj",  block.mlp.c_proj),
    ]

def flat_grads(module, device, dtype=torch.float32):
    out = []
    for p in module.parameters():
        if not p.requires_grad:
            continue
        out.append(
            p.grad.detach().reshape(-1).to(dtype) if p.grad is not None
            else torch.zeros(p.numel(), device=device, dtype=dtype)
        )
    return out

# ── dataset (load once) ───────────────────────────────────────────────────────
print("Loading dataset...")
# dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train[:50]")

raw = load_dataset("karpathy/tiny_shakespeare", split="train")
full_text = raw[0]["text"]

# Chunk into sequences of ~512 chars, take first NUM_EXAMPLES
CHUNK_CHARS = 512
chunks = [
    {"text": full_text[i : i + CHUNK_CHARS]}
    for i in range(0, len(full_text), CHUNK_CHARS)
]
chunks = chunks[:NUM_EXAMPLES]  # same as train[:50] logic

from torch.utils.data import Dataset

class ListDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

dataloader = DataLoader(ListDataset(chunks), batch_size=1, shuffle=False)
enc = tiktoken.get_encoding("gpt2")

# ── main loop ─────────────────────────────────────────────────────────────────
for step in CKPT_STEPS:
    tag       = str(step).zfill(STEP_WIDTH)
    ckpt_path = os.path.join(CKPT_DIR, f"{step}_ckpt.pt")

    # skip if checkpoint missing OR both .npy files already exist
    bn_path = os.path.join(NPY_DIR, f"block_norms_step{tag}.npy")
    pn_path = os.path.join(NPY_DIR, f"part_norms_step{tag}.npy")
    if os.path.exists(bn_path) and os.path.exists(pn_path):
        print(f"[skip] step={step} already done")
        continue
    if not os.path.exists(ckpt_path):
        print(f"[skip] {ckpt_path} not found")
        continue

    print(f"\n{'='*60}\n  step={step}  ({ckpt_path})\n{'='*60}")

    # ── load model ────────────────────────────────────────────────────────────
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    config = GPTConfig(**checkpoint["model_args"])
    if hasattr(config, "flash"):
        config.flash = False

    model = GPT(config)
    sd = checkpoint.get("model", checkpoint)
    sd = {k.replace("_orig_mod.", ""): v for k, v in sd.items()}
    model.load_state_dict(sd, strict=False)
    model.to(device).eval()

    vocab_size       = int(checkpoint["model_args"]["vocab_size"])
    model_block_size = int(checkpoint["model_args"]["block_size"])
    block_size       = min(BLOCK_SIZE_CAP, model_block_size)

    transformer_blocks = model.transformer.h[:NUM_LAYERS]
    num_blocks = len(transformer_blocks)

    # ── part metadata ─────────────────────────────────────────────────────────
    part_meta, cursor = [], 0
    for bi, block in enumerate(transformer_blocks):
        for pname, pmod in iter_parts(block):
            n = sum(p.numel() for p in pmod.parameters() if p.requires_grad)
            part_meta.append(dict(block=bi, part=pname, size=n, sl=slice(cursor, cursor + n)))
            cursor += n
    total_params = cursor

    # ── fake param + AdamGram ─────────────────────────────────────────────────
    fake_param = torch.nn.Parameter(
        torch.zeros(total_params, device=device, dtype=torch.float32),
        requires_grad=True,
    )
    ada_gram_fr = AdamGram(
        [fake_param],
        lr=1e-5, eps=1e-4, weight_decay=0,
        max_rank=MAX_RANK,
        task=f"ckpt_{tag}",
        enable_logging=False, save_matrix=False,
    )

    # ── accumulate Fisher ─────────────────────────────────────────────────────
    for batch_idx, batch in enumerate(tqdm(dataloader, desc=f"step {step}", total=NUM_EXAMPLES)):
        if batch_idx >= NUM_EXAMPLES:
            break
        text = batch["text"][0]
        x, y = make_xy_from_text(text, vocab_size, block_size, device, enc)

        model.zero_grad(set_to_none=True)
        logits, loss = model(x, targets=y)
        loss.backward()

        all_grads = []
        for block in transformer_blocks:
            for _, pmod in iter_parts(block):
                all_grads.extend(flat_grads(pmod, device=device))

        fake_param.grad = torch.cat(all_grads)
        ada_gram_fr.step()
        model.zero_grad(set_to_none=True)
        fake_param.grad = None

    # ── extract P, Q ──────────────────────────────────────────────────────────
    state = ada_gram_fr.state[fake_param]
    P_full = state["P"].clone()
    Q_full = state["Q"].clone()

    P_parts = {(m["block"], m["part"]): P_full[m["sl"], :] for m in part_meta}
    Q_parts = {(m["block"], m["part"]): Q_full[m["sl"], :] for m in part_meta}

    # ── block norms ───────────────────────────────────────────────────────────
    block_slices = []
    for bi in range(num_blocks):
        s0 = min(m["sl"].start for m in part_meta if m["block"] == bi)
        s1 = max(m["sl"].stop  for m in part_meta if m["block"] == bi)
        block_slices.append(slice(s0, s1))

    block_norms = torch.zeros((num_blocks, num_blocks))
    for i in range(num_blocks):
        Pi = P_full[block_slices[i], :]
        for j in range(num_blocks):
            Qj = Q_full[block_slices[j], :]
            block_norms[i, j] = frob_norm_lowrank(Pi, Qj)
    block_norms_np = block_norms.numpy()

    # ── part norms ────────────────────────────────────────────────────────────
    part_keys  = [(m["block"], m["part"]) for m in part_meta]
    K          = len(part_keys)
    part_norms = torch.zeros((K, K))
    for i in tqdm(range(K), desc="Part norms", leave=False):
        Pi = P_parts[part_keys[i]]
        if Pi.numel() == 0: continue
        for j in range(K):
            Qj = Q_parts[part_keys[j]]
            if Qj.numel() == 0: continue
            part_norms[i, j] = frob_norm_lowrank(Pi, Qj)
    part_norms_np = part_norms.numpy()

    # ── save ──────────────────────────────────────────────────────────────────
    np.save(bn_path, block_norms_np)
    np.save(pn_path, part_norms_np)
    print(f"[saved] {bn_path}")
    print(f"[saved] {pn_path}")

    del model, fake_param, ada_gram_fr, P_full, Q_full
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\nDone! All checkpoints processed.")


In [7]:
import gc
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
import tiktoken

from adagram_optimizers.AdamGram import AdamGram
from model import GPT, GPTConfig
import matplotlib.colors as mcolors

# ── config ────────────────────────────────────────────────────────────────────
CKPT_DIR        = "out_adagramps_bs512_lr1e-2_rank5"
CKPT_STEPS      = list(range(250, 5001, 250))
SPLIT_MODE      = "fine"
PLOT_DIR        = f"plots_{SPLIT_MODE}_all_ckpts"
NUM_LAYERS      = 6
NUM_EXAMPLES    = 50
BLOCK_SIZE_CAP  = 128
MAX_RANK        = 5
STEP_WIDTH      = len(str(max(CKPT_STEPS)))

os.makedirs(PLOT_DIR, exist_ok=True)
device = torch.device("cpu")

# ── helpers ───────────────────────────────────────────────────────────────────
def make_xy_from_text(text, vocab_size, block_size, device, enc):
    ids = enc.encode(text)
    ids = [min(max(int(t), 0), vocab_size - 1) for t in ids]
    ids = ids[: block_size + 1]
    if len(ids) < 2:
        ids = ids + [0]
    x = torch.tensor(ids[:-1], dtype=torch.long, device=device)[None, :]
    y = torch.tensor(ids[1:],  dtype=torch.long, device=device)[None, :]
    return x, y

def iter_parts(block):
    return [
        ("ln_1",        block.ln_1),
        ("attn.c_attn", block.attn.c_attn),
        ("attn.c_proj", block.attn.c_proj),
        ("ln_2",        block.ln_2),
        ("mlp.c_fc",    block.mlp.c_fc),
        ("mlp.c_proj",  block.mlp.c_proj),
    ]

def flat_grads(module, device, dtype=torch.float32):
    out = []
    for p in module.parameters():
        if not p.requires_grad:
            continue
        out.append(
            p.grad.detach().reshape(-1).to(dtype) if p.grad is not None
            else torch.zeros(p.numel(), device=device, dtype=dtype)
        )
    return out

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print(f"[saved] {path}")
    plt.close()

PART_NAMES = ["ln_1", "attn.c_attn", "attn.c_proj", "ln_2", "mlp.c_fc", "mlp.c_proj"]

def diagonality_score(P_ref, Q_j):
    with torch.no_grad():
        n_diag = min(P_ref.shape[0], Q_j.shape[0])
        if n_diag == 0:
            return 0.0
        diag_sq_sum = (P_ref[:n_diag] * Q_j[:n_diag]).sum(dim=1).pow(2).sum()
        G_P = P_ref.T @ P_ref
        G_Q = Q_j.T  @ Q_j
        frob_sq = torch.trace(G_P @ G_Q)
        if frob_sq < 1e-30:
            print("zero block")
            return diag_sq_sum
        return (diag_sq_sum / frob_sq).item()

def plot_heatmap(score_matrix, part_labels, num_blocks, step, current_rank, path):
    K         = len(part_labels)
    num_parts = len(PART_NAMES)
    fig_sz    = max(12, K * 0.4)

    # avoid log(0): clip to a small floor before normalizing
    eps    = 1e-6
    data   = np.clip(score_matrix, eps, 1.0)
    norm   = mcolors.LogNorm(vmin=eps, vmax=1.0)

    fig, ax = plt.subplots(figsize=(fig_sz, fig_sz * 0.85))
    im = ax.imshow(data, norm=norm, cmap="viridis", aspect="auto")

    cbar = plt.colorbar(im, ax=ax, label="Diagonality score ρ  (log scale)")
    # show clean tick labels on the log colorbar
    cbar.set_ticks([1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0])
    cbar.set_ticklabels(["≤1e-6", "1e-5", "1e-4", "1e-3", "1e-2", "0.1", "1.0"])

    ax.set_xticks(range(K))
    ax.set_yticks(range(K))
    ax.set_xticklabels(part_labels, rotation=90, fontsize=7)
    ax.set_yticklabels(part_labels, fontsize=7)
    ax.set_xlabel("j  (Q slice)")
    ax.set_ylabel("i  (P slice)")
    ax.set_title(
        f"Diagonality score  ρ(i,j) = diag_score(P_i @ Q_j.T)  [log scale]\n"
        f"step={step},  rank={current_rank}"
    )

    for b in range(1, num_blocks):
        ax.axhline(b * num_parts - 0.5, color="red", linewidth=0.8, linestyle="--")
        ax.axvline(b * num_parts - 0.5, color="red", linewidth=0.8, linestyle="--")

    savefig(path)

# ── dataset (load once) ───────────────────────────────────────────────────────
print("Loading dataset...")
with open("data/shakespeare_char/input.txt", "r") as f:
    raw_text = f.read()

chunk_size = 512
chunks = [raw_text[i:i + chunk_size] for i in range(0, len(raw_text), chunk_size)]
chunks = chunks[:NUM_EXAMPLES]
dataloader = [{"text": [c]} for c in chunks]

enc = tiktoken.get_encoding("gpt2")

# ── main loop over checkpoints ────────────────────────────────────────────────
for step in CKPT_STEPS:
    tag        = str(step).zfill(STEP_WIDTH)
    ckpt_path  = os.path.join(CKPT_DIR, f"{step}_ckpt.pt")
    npy_path   = os.path.join(PLOT_DIR, f"diagonality_matrix_step{tag}.npy")
    plot_path  = os.path.join(PLOT_DIR, f"diagonality_heatmap_step{tag}.png")

    # ── derive static part labels (don't need model for this) ─────────────────
    num_blocks  = NUM_LAYERS
    part_keys   = [(b, p) for b in range(num_blocks) for p in PART_NAMES]
    part_labels = [f"B{b}:{p}" for b, p in part_keys]
    K           = len(part_keys)

    # ── fast path: npy already exists → just replot ───────────────────────────
    if os.path.exists(npy_path):
        print(f"[cache] loading {npy_path}")
        score_matrix = np.load(npy_path)
        plot_heatmap(score_matrix, part_labels, num_blocks, step,
                     current_rank="?", path=plot_path)
        del score_matrix
        continue

    # ── slow path: checkpoint must exist ──────────────────────────────────────
    if not os.path.exists(ckpt_path):
        print(f"[skip] {ckpt_path} not found")
        continue

    print(f"\n{'='*60}")
    print(f"  Processing checkpoint step={step}  ({ckpt_path})")
    print(f"{'='*60}")

    # ── load model ────────────────────────────────────────────────────────────
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    config = GPTConfig(**checkpoint["model_args"])
    if hasattr(config, "flash"):
        config.flash = False

    model = GPT(config)
    sd = checkpoint.get("model", checkpoint)
    sd = {k.replace("_orig_mod.", ""): v for k, v in sd.items()}
    model.load_state_dict(sd, strict=False)
    model.to(device).eval()

    vocab_size       = int(checkpoint["model_args"]["vocab_size"])
    model_block_size = int(checkpoint["model_args"]["block_size"])
    block_size       = min(BLOCK_SIZE_CAP, model_block_size)

    transformer_blocks = model.transformer.h[:NUM_LAYERS]

    # ── part metadata ─────────────────────────────────────────────────────────
    part_meta, cursor = [], 0
    for bi, block in enumerate(transformer_blocks):
        for pname, pmod in iter_parts(block):
            n = sum(p.numel() for p in pmod.parameters() if p.requires_grad)
            part_meta.append(dict(block=bi, part=pname, size=n, sl=slice(cursor, cursor + n)))
            cursor += n
    total_params = cursor

    # ── fake param + AdamGram ─────────────────────────────────────────────────
    fake_param = torch.nn.Parameter(
        torch.zeros(total_params, device=device, dtype=torch.float32),
        requires_grad=True,
    )
    ada_gram_fr = AdamGram(
        [fake_param],
        lr=1e-5, eps=1e-4, weight_decay=0,
        max_rank=MAX_RANK,
        task=f"ckpt_{tag}",
        enable_logging=False, save_matrix=False,
    )

    # ── accumulate Fisher ─────────────────────────────────────────────────────
    for batch_idx, batch in enumerate(tqdm(dataloader, desc=f"Step {step}", total=NUM_EXAMPLES)):
        if batch_idx >= NUM_EXAMPLES:
            break
        text = batch["text"][0]
        x, y = make_xy_from_text(text, vocab_size, block_size, device, enc)

        model.zero_grad(set_to_none=True)
        logits, loss = model(x, targets=y)
        loss.backward()

        all_grads = []
        for block in transformer_blocks:
            for _, pmod in iter_parts(block):
                all_grads.extend(flat_grads(pmod, device=device))

        full_grad = torch.cat(all_grads)
        del all_grads
        fake_param.grad = full_grad
        ada_gram_fr.step()
        del full_grad
        model.zero_grad(set_to_none=True)
        fake_param.grad = None
        gc.collect()

    # ── extract P, Q ──────────────────────────────────────────────────────────
    state = ada_gram_fr.state[fake_param]
    P_full, Q_full = state["P"].clone(), state["Q"].clone()
    current_rank = P_full.shape[1] if P_full.dim() > 1 else 1

    del model, ada_gram_fr, fake_param
    gc.collect()

    P_parts = {key: P_full[m["sl"], :] for key, m in zip(part_keys, part_meta)}
    Q_parts = {key: Q_full[m["sl"], :] for key, m in zip(part_keys, part_meta)}
    del P_full, Q_full
    gc.collect()

    # ── compute K×K diagonality score matrix ──────────────────────────────────
    score_matrix = np.zeros((K, K))
    for i, key_i in enumerate(tqdm(part_keys, desc="Diagonality matrix")):
        P_i = P_parts[key_i]
        for j, key_j in enumerate(part_keys):
            score_matrix[i, j] = diagonality_score(P_i, Q_parts[key_j])

    np.save(npy_path, score_matrix)
    print(f"[saved] {npy_path}")

    plot_heatmap(score_matrix, part_labels, num_blocks, step, current_rank, plot_path)

    del P_parts, Q_parts, score_matrix
    gc.collect()

print("\nDone! All checkpoints processed.")

Loading dataset...

  Processing checkpoint step=250  (out_adagramps_bs512_lr1e-2_rank5/250_ckpt.pt)
number of parameters: 10.65M


Diagonality matrix: 100%|██████████| 36/36 [00:26<00:00,  1.38it/s]


[saved] plots_fine_all_ckpts/diagonality_matrix_step0250.npy
[saved] plots_fine_all_ckpts/diagonality_heatmap_step0250.png

  Processing checkpoint step=500  (out_adagramps_bs512_lr1e-2_rank5/500_ckpt.pt)
number of parameters: 10.65M


Diagonality matrix: 100%|██████████| 36/36 [00:21<00:00,  1.66it/s]


[saved] plots_fine_all_ckpts/diagonality_matrix_step0500.npy
[saved] plots_fine_all_ckpts/diagonality_heatmap_step0500.png

  Processing checkpoint step=750  (out_adagramps_bs512_lr1e-2_rank5/750_ckpt.pt)
number of parameters: 10.65M


...

[saved] plots_fine_all_ckpts/diagonality_matrix_step2250.npy
[saved] plots_fine_all_ckpts/diagonality_heatmap_step2250.png

  Processing checkpoint step=2500  (out_adagramps_bs512_lr1e-2_rank5/2500_ckpt.pt)
number of parameters: 10.65M


Diagonality matrix: 100%|██████████| 36/36 [03:01<00:00,  5.04s/it]


[saved] plots_fine_all_ckpts/diagonality_matrix_step2500.npy
[saved] plots_fine_all_ckpts/diagonality_heatmap_step2500.png

  Processing checkpoint step=2750  (out_adagramps_bs512_lr1e-2_rank5/2750_ckpt.pt)
number of parameters: 10.65M


Step 2750:  84%|████████▍ | 42/50 [11:19<02:09, 16.18s/it]


KeyboardInterrupt: 

In [4]:
part_meta

[{'block': 0, 'part': 'ln_1', 'size': 384, 'sl': slice(0, 384, None)},
 {'block': 0,
  'part': 'attn.c_attn',
  'size': 442368,
  'sl': slice(384, 442752, None)},
 {'block': 0,
  'part': 'attn.c_proj',
  'size': 147456,
  'sl': slice(442752, 590208, None)},
 {'block': 0, 'part': 'ln_2', 'size': 384, 'sl': slice(590208, 590592, None)},
 {'block': 0,
  'part': 'mlp.c_fc',
  'size': 589824,
  'sl': slice(590592, 1180416, None)},
 {'block': 0,
  'part': 'mlp.c_proj',
  'size': 589824,
  'sl': slice(1180416, 1770240, None)},
 {'block': 1,
  'part': 'ln_1',
  'size': 384,
  'sl': slice(1770240, 1770624, None)},
 {'block': 1,
  'part': 'attn.c_attn',
  'size': 442368,
  'sl': slice(1770624, 2212992, None)},
 {'block': 1,
  'part': 'attn.c_proj',
  'size': 147456,
  'sl': slice(2212992, 2360448, None)},
 {'block': 1,
  'part': 'ln_2',
  'size': 384,
  'sl': slice(2360448, 2360832, None)},
 {'block': 1,
  'part': 'mlp.c_fc',
  'size': 589824,
  'sl': slice(2360832, 2950656, None)},
 {'block': 1

In [4]:
# ── helpers ───────────────────────────────────────────────────────────────────

def reconstruct_F_block(P_i, Q_j):
    """
    Reconstruct the cross-block Fisher submatrix F[i,j] = P_i @ Q_j.T
    Returns a 2D tensor of shape (size_i, size_j).
    """
    return P_i @ Q_j.T  # (n_i, rank) x (rank, n_j) -> (n_i, n_j)


def count_diag_offdiag_cross_elements(P_parts, Q_parts, part_meta, ref_block, ref_part):
    """
    For the given (ref_block, ref_part), reconstruct all submatrices
    F[ref_i, j] = P_ref @ Q_j.T and collect their elements into three buckets:
      - diag:      j is same block AND same part  → diagonal elements only (min dim)
      - off_diag:  j is same block, different part → all elements of that submatrix
      - cross:     j is different block            → all elements of that submatrix
    Returns three flat numpy arrays of scalar values.
    """
    part_keys = [(m["block"], m["part"]) for m in part_meta]
    P_ref = P_parts[(ref_block, ref_part)]  # (n_ref, rank)

    diag_elems     = []
    off_diag_elems = []
    cross_elems    = []

    for (bj, pj) in part_keys:
        Q_j = Q_parts[(bj, pj)]             # (n_j, rank)
        F_sub = reconstruct_F_block(P_ref, Q_j)  # (n_ref, n_j)

        if bj == ref_block and pj == ref_part:
            # True diagonal block: take only the diagonal elements
            n_diag = min(F_sub.shape)
            diag_elems.append(torch.diag(F_sub[:n_diag, :n_diag]).cpu().numpy())
        elif bj == ref_block:
            # Same block, different part: all off-diagonal elements
            off_diag_elems.append(F_sub.reshape(-1).cpu().numpy())
        else:
            # Different block entirely
            cross_elems.append(F_sub.reshape(-1).cpu().numpy())

    diag_elems     = np.concatenate(diag_elems)     if diag_elems     else np.array([])
    off_diag_elems = np.concatenate(off_diag_elems) if off_diag_elems else np.array([])
    cross_elems    = np.concatenate(cross_elems)    if cross_elems    else np.array([])

    return diag_elems, off_diag_elems, cross_elems



# ── plot per (block, part): diag / off-diag / cross element counts ────────
unique_blocks = sorted(set(m["block"] for m in part_meta))
part_names    = [p for _, p in iter_parts(transformer_blocks[0])]
for ref_block in unique_blocks:
    diag_counts     = []
    off_diag_counts = []
    cross_counts    = []
    x_labels        = []
    for ref_part in part_names:
        if (ref_block, ref_part) not in P_parts:
            continue
        d_elems, od_elems, c_elems = count_diag_offdiag_cross_elements(
            P_parts, Q_parts, part_meta, ref_block, ref_part
        )
        diag_counts.append(len(d_elems))
        off_diag_counts.append(len(od_elems))
        cross_counts.append(len(c_elems))
        x_labels.append(ref_part)
    x_pos = np.arange(len(x_labels))
    width = 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x_pos - width, diag_counts,     width, label="Same block, same part (diag)",  color="#1f77b4")
    ax.bar(x_pos,         off_diag_counts, width, label="Same block, diff part",          color="#ff7f0e")
    ax.bar(x_pos + width, cross_counts,    width, label="Cross-block",                    color="#2ca02c")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, rotation=30, ha="right")
    ax.set_xlabel("Part name")
    ax.set_ylabel("Number of elements")
    ax.set_title(
        f"Per-(block, part) diag vs off-diag counts\n"
        f"step={step}, block={ref_block}"
    )
    ax.legend()
    ax.yaxis.grid(True, linestyle="--", linewidth=0.6)
    ax.set_axisbelow(True)
    savefig(os.path.join(PLOT_DIR, f"diag_offdiag_counts_step{tag}_block{ref_block}.png"))

NameError: name 'P_parts' is not defined

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PASS 2 — load all saved .npy files, compute global bounds, plot with fixed scale
# ══════════════════════════════════════════════════════════════════════════════
import os
import numpy as np
import torch
from tqdm import tqdm
import tiktoken
from datasets import load_dataset
from torch.utils.data import DataLoader

from adagram_optimizers.AdamGram import AdamGram
import matplotlib.pyplot as plt
import seaborn as sns

from model import GPT, GPTConfig

print("\nPass 2: plotting with unified color scale...")
eps = 1e-30
PLOT_DIR = "adamw_all_plots"

CKPT_DIR       = "out_adamw_bs16_lr1e-4"
CKPT_STEPS     = list(range(250, 5001, 250))
SPLIT_MODE     = "fine"
NPY_DIR        = f"adamw_npys_{SPLIT_MODE}_all_ckpts"
NUM_LAYERS     = 6
NUM_EXAMPLES   = 50
BLOCK_SIZE_CAP = 128
MAX_RANK       = 5
STEP_WIDTH     = len(str(max(CKPT_STEPS)))
os.makedirs(PLOT_DIR, exist_ok=True)

PART_NAMES = ["ln_1", "attn.c_attn", "attn.c_proj", "ln_2", "mlp.c_fc", "mlp.c_proj"]

def make_part_labels(num_blocks):
    return [f"B{b}:{p}" for b in range(num_blocks) for p in PART_NAMES]

# per-(block, part) diag / off-diag stats for part matrix pn
def per_blockpart_diag_stats(pn: np.ndarray, num_blocks: int, part_names: list):
    """
    For each (block b, part p), aggregate:
      - diag:    pn[idx, idx] with idx = b*P + p
      - within:  pn[idx, j] where j in same block (same b) but different part
      - cross:   pn[idx, j] where j in different block (bj != b)
    Returns arrays of shape (num_blocks, P).
    """
    P = len(part_names)
    K = pn.shape[0]
    assert K == num_blocks * P

    diag_sums         = np.zeros((num_blocks, P), dtype=pn.dtype)
    within_off_sums   = np.zeros((num_blocks, P), dtype=pn.dtype)
    cross_sums        = np.zeros((num_blocks, P), dtype=pn.dtype)
    diag_counts       = np.zeros((num_blocks, P), dtype=int)
    within_off_counts = np.zeros((num_blocks, P), dtype=int)
    cross_counts      = np.zeros((num_blocks, P), dtype=int)

    for i in range(K):
        bi, pi = divmod(i, P)   # block index, part index
        for j in range(K):
            bj, pj = divmod(j, P)
            val = pn[i, j]
            if bi == bj and pi == pj:      # true diagonal
                diag_sums[bi, pi] += val
                diag_counts[bi, pi] += 1
            elif bi == bj:                 # same block, different part
                within_off_sums[bi, pi] += val
                within_off_counts[bi, pi] += 1
            else:                          # cross-block
                cross_sums[bi, pi] += val
                cross_counts[bi, pi] += 1

    return diag_counts, within_off_counts, cross_counts, diag_sums, within_off_sums, cross_sums

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print(f"[saved] {path}")
    plt.close()

# ── load all arrays ───────────────────────────────────────────────────────────
all_block   = {}
all_logblock = {}
all_part    = {}
all_logpart = {}

for step in CKPT_STEPS:
    tag = str(step).zfill(STEP_WIDTH)
    bp = os.path.join(NPY_DIR, f"block_norms_step{tag}.npy")
    pp = os.path.join(NPY_DIR, f"part_norms_step{tag}.npy")
    if not os.path.exists(bp):
        print(f"[skip] {bp} not found")
        continue
    bn = np.load(bp)
    pn = np.load(pp)
    all_block[step]    = bn
    all_logblock[step] = np.log10(bn + eps)
    all_part[step]     = pn
    all_logpart[step]  = np.log10(pn + eps)

# ── global bounds ─────────────────────────────────────────────────────────────
def global_bounds(d):
    vals = np.concatenate([v.ravel() for v in d.values()])
    return vals.min(), vals.max()

b_vmin,  b_vmax  = global_bounds(all_block)
lb_vmin, lb_vmax = global_bounds(all_logblock)
p_vmin,  p_vmax  = global_bounds(all_part)
lp_vmin, lp_vmax = global_bounds(all_logpart)

print(f"Block linear  vmin={b_vmin:.3e}  vmax={b_vmax:.3e}")
print(f"Block log10   vmin={lb_vmin:.3f}  vmax={lb_vmax:.3f}")
print(f"Part  linear  vmin={p_vmin:.3e}  vmax={p_vmax:.3e}")
print(f"Part  log10   vmin={lp_vmin:.3f}  vmax={lp_vmax:.3f}")

# ── plot all checkpoints with fixed scale ─────────────────────────────────────
for step in tqdm(CKPT_STEPS, desc="Plotting"):
    if step not in all_block:
        continue
    tag          = str(step).zfill(STEP_WIDTH)
    bn           = all_block[step]
    num_blocks   = bn.shape[0]
    blabels      = [f"B{i}" for i in range(num_blocks)]
    title_suffix = f"step={step}"

    # parts-related data
    pn     = all_part[step]
    K      = pn.shape[0]
    num_blocks_for_parts = K // len(PART_NAMES)
    assert num_blocks_for_parts == num_blocks  # sanity check
    part_labels = make_part_labels(num_blocks_for_parts)
    fig_sz = (max(10, K * 0.35), max(8, K * 0.35))

    # ── per-(block, part) stats ───────────────────────────────────────────────
    (bp_diag_counts, bp_within_counts, bp_cross_counts,
     bp_diag_sums, bp_within_sums, bp_cross_sums) = per_blockpart_diag_stats(
         pn, num_blocks=num_blocks_for_parts, part_names=PART_NAMES
    )

    # one diagram per block
    x_parts = np.arange(len(PART_NAMES))
    width = 0.25
    for b in range(num_blocks):
        # counts for this block
        diag_c   = bp_diag_counts[b]    # shape (P,)
        within_c = bp_within_counts[b]
        cross_c  = bp_cross_counts[b]

        plt.figure(figsize=(8, 4))
        plt.bar(x_parts - width, diag_c,   width, label="Same block, same part (diag)")
        plt.bar(x_parts,         within_c, width, label="Same block, diff part")
        plt.bar(x_parts + width, cross_c,  width, label="Cross-block")
        plt.xticks(x_parts, PART_NAMES, rotation=30, ha="right")
        plt.xlabel("Part name")
        plt.ylabel("Number of elements")
        plt.title(f"Per-(block, part) diag vs off-diag counts\n{title_suffix}, block={b}")
        plt.legend()
        plt.grid(True, axis="y", linestyle="--", linewidth=0.5)
        savefig(os.path.join(PLOT_DIR, f"per_block{b}_diag_offdiag_counts_step{tag}.png"))

        # sums for this block
        diag_s   = bp_diag_sums[b]
        within_s = bp_within_sums[b]
        cross_s  = bp_cross_sums[b]

        plt.figure(figsize=(8, 4))
        plt.bar(x_parts - width, diag_s,   width, label="Same block, same part (diag)")
        plt.bar(x_parts,         within_s, width, label="Same block, diff part")
        plt.bar(x_parts + width, cross_s,  width, label="Cross-block")
        plt.xticks(x_parts, PART_NAMES, rotation=30, ha="right")
        plt.yscale("log")
        plt.xlabel("Part name")
        plt.ylabel("Sum of norms")
        plt.title(f"Per-(block, part) diag vs off-diag norm sums\n{title_suffix}, block={b}")
        plt.legend()
        plt.grid(True, which="both", linestyle="--", linewidth=0.5)
        savefig(os.path.join(PLOT_DIR, f"per_block{b}_diag_offdiag_sums_step{tag}.png"))

    # 1) block linear heatmap
    plt.figure(figsize=(7, 6))
    ax = sns.heatmap(bn, annot=num_blocks <= 12, fmt=".2e", square=True, cbar=True,
                     xticklabels=blabels, yticklabels=blabels,
                     vmin=b_vmin, vmax=b_vmax)
    ax.set_title(f"Block-to-Block Frobenius norms\n{title_suffix}")
    ax.set_xlabel("Block j (Q)"); ax.set_ylabel("Block i (P)")
    savefig(os.path.join(PLOT_DIR, f"block_block_norms_step{tag}.png"))

    # 2) block log10 heatmap
    plt.figure(figsize=(7, 6))
    ax = sns.heatmap(all_logblock[step], annot=num_blocks <= 12, fmt=".2f", square=True, cbar=True,
                     xticklabels=blabels, yticklabels=blabels,
                     vmin=lb_vmin, vmax=lb_vmax)
    ax.set_title(f"log10 Block-to-Block norms\n{title_suffix}")
    ax.set_xlabel("Block j (Q)"); ax.set_ylabel("Block i (P)")
    savefig(os.path.join(PLOT_DIR, f"block_block_norms_log10_step{tag}.png"))

    # 3) block summaries
    row_sum = bn.sum(axis=1)
    col_sum = bn.sum(axis=0)
    plt.figure(figsize=(8, 4))
    plt.plot(range(num_blocks), row_sum, marker="o", label="Row-sum (P_i, all Q_j)")
    plt.plot(range(num_blocks), col_sum, marker="s", label="Col-sum (Q_j, all P_i)")
    plt.xticks(range(num_blocks), blabels)
    plt.yscale("log"); plt.grid(True, which="both", linestyle="--", linewidth=0.5)
    plt.title(f"Block interaction strength\n{title_suffix}")
    plt.xlabel("Block index"); plt.ylabel("Sum of norms"); plt.legend()
    savefig(os.path.join(PLOT_DIR, f"block_summaries_step{tag}.png"))

    # 4) part linear heatmap
    plt.figure(figsize=fig_sz)
    ax = sns.heatmap(pn, square=True, cbar=True,
                     xticklabels=part_labels, yticklabels=part_labels,
                     vmin=p_vmin, vmax=p_vmax)
    ax.set_title(f"Part-to-Part Frobenius norms\n{title_suffix}")
    ax.set_xlabel("Part j (Q)"); ax.set_ylabel("Part i (P)")
    plt.xticks(rotation=90); plt.yticks(rotation=0)
    savefig(os.path.join(PLOT_DIR, f"part_norms_step{tag}.png"))

    # 5) part log10 heatmap
    plt.figure(figsize=fig_sz)
    ax = sns.heatmap(all_logpart[step], square=True, cbar=True,
                     xticklabels=part_labels, yticklabels=part_labels,
                     vmin=lp_vmin, vmax=lp_vmax)
    ax.set_title(f"log10 Part-to-Part norms\n{title_suffix}")
    ax.set_xlabel("Part j (Q)"); ax.set_ylabel("Part i (P)")
    plt.xticks(rotation=90); plt.yticks(rotation=0)
    savefig(os.path.join(PLOT_DIR, f"part_norms_log10_step{tag}.png"))

print("Done.")

In [ ]:
from pathlib import Path
output_folder = Path(f"adagram_attention_blocks_eq_hist")
output_folder.mkdir(exist_ok=True)
print(f"✓ Created folder: {output_folder}")

import os
import torch
import numpy as np
import pandas as pd
import datashader as ds
import datashader.transfer_functions as tf
from datashader.utils import export_image
from colorcet import fire
from pathlib import Path
from matplotlib import cm
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy import stats
from mpl_toolkits.axes_grid1 import ImageGrid
from datashader.mpl_ext import dsshow, alpha_colormap

# -----------------------------
# Controls
# -----------------------------
plot_width = 2000
plot_height = 2000
max_entries_per_block = 100_000_000
value_agg = ds.mean("v")
shade_how = "eq_hist"
figsize = (10, 12)


def save_block_with_grid(block_name, block_agg, block_v, output_path, figsize=(6, 5)):
    """Save a single block using ImageGrid style with colorbar from tf.shade"""
    fig = plt.figure(figsize=figsize, facecolor='black')
    
    # Create a single-axis grid with colorbar
    grid = ImageGrid(fig, 111, nrows_ncols=(1, 1), 
                     axes_pad=0.5, share_all=True,
                     cbar_location="right", cbar_mode="single", 
                     cbar_size="5%", cbar_pad="2%")
    
    # Create the shaded image
    img = tf.shade(block_agg, cmap=fire, how='eq_hist')
    
    # Display the image on the grid axis
    artist = grid[0].imshow(img, origin='upper', interpolation='antialiased')
    
    # Create a colorbar that matches the data
    data_flat = block_v[~np.isnan(block_v)]
    if len(data_flat) > 0:
        # Create a ScalarMappable without explicit norm (uses default linear norm)
        sm = ScalarMappable(cmap=fire)
        sm.set_array(data_flat)  # This automatically sets vmin/vmax from data range
        
        # Add colorbar
        cbar = plt.colorbar(sm, cax=grid.cbar_axes[0])
        
        # Style the colorbar
        cbar.ax.tick_params(colors='white', labelsize=8)
        cbar.set_label('Attention Value\n(linear)', rotation=270, labelpad=20, 
                       color='white', fontsize=10)
        
        # Add percentile ticks
        percentiles = [1, 25, 50, 75, 99]
        perc_values = np.percentile(data_flat, percentiles)
        cbar.ax.set_yticks(perc_values)
        cbar.ax.set_yticklabels([f'{p}%' for p in percentiles])
        
        # Add stats text
        stats_text = f'min={data_flat.min():.3f}\nmax={data_flat.max():.3f}\nmean={data_flat.mean():.3f}'
        grid[0].text(0.02, 0.98, stats_text, transform=grid[0].transAxes, 
                     fontsize=8, color='cyan', verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
    
    grid[0].set_title(block_name, color='white', fontsize=12, pad=10)
    grid[0].set_axis_off()
    
    plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='black')
    plt.close(fig)
# -----------------------------
# Helpers: emit sampled entries from A = Pi @ Qi.T without forming A
# -----------------------------
def emit_block_samples(Pi, Qi, row0, col0, n_i, max_entries, seed=0):  # ← FIXED: added n_i param
    """Pi,Qi: torch tensors (n, r) on CPU or GPU. Returns x, y, v samples."""
    n, r = Pi.shape
    m = n * n
    rng = np.random.default_rng(seed)

    if m <= max_entries:
        A = (Qi @ Pi.T).detach().cpu().numpy()
        rr, cc = np.indices((n, n))
        rr = rr.reshape(-1).astype(np.int64)
        cc = cc.reshape(-1).astype(np.int64)
        vv = A.reshape(-1).astype(np.float32)
    else:
        idx = rng.integers(0, m, size=max_entries, endpoint=False, dtype=np.int64)
        rr = (idx // n).astype(np.int64)
        cc = (idx % n).astype(np.int64)
        rr_t = torch.from_numpy(rr).to(Pi.device)
        cc_t = torch.from_numpy(cc).to(Pi.device)
        vv = torch.sum(Pi[rr_t] * Qi[cc_t], dim=1).detach().cpu().numpy().astype(np.float32)

    y = (n_i - 1 - rr) + row0  # Flip vertically before plotting
    x = cc + col0
    return x, y, vv

# -----------------------------
# Build and save each block individually WITH COLORBAR
# -----------------------------
xs, ys, vs = [], [], []
offset = 0

for i, m in enumerate(part_meta):
    layer_name = m["part"]
    sl = m["sl"]
    head_number = m["block"]
    Pi = P_full[sl, :]
    Qi = Q_full[sl, :]
    n_i = Pi.shape[0]
    if n_i == 0:
        continue

    # Generate samples for this block
    x, y, v = emit_block_samples(
        Pi, Qi, row0=offset, col0=offset, n_i=n_i,
        max_entries=max_entries_per_block, seed=1234 + i
    )

    # DataFrame for datashader
    block_df = pd.DataFrame({
        "x": x.astype(np.int64), 
        "y": y.astype(np.int64), 
        "v": v.astype(np.float32)
    })

    # Create canvas and aggregate
    block_canvas = ds.Canvas(plot_width=800, plot_height=800,
                           x_range=(offset, offset + n_i),
                           y_range=(offset, offset + n_i))
    
    block_agg = block_canvas.points(block_df, "x", "y", value_agg)

    # Save with grid style
    block_filename = f"block_{i}_layer_{layer_name}_size_{n_i}_head_num_{head_number}"
    block_display_name = f"Layer {layer_name} • Head {head_number} • n={n_i}"

    # Create figure with ImageGrid
    fig = plt.figure(figsize=(6, 5), facecolor='black')
    
    # Create a single-axis grid with colorbar
    grid = ImageGrid(fig, 111, nrows_ncols=(1, 1), 
                     axes_pad=0.5, share_all=True,
                     cbar_location="right", cbar_mode="single", 
                     cbar_size="5%", cbar_pad="2%")
    
    # Create the dsshow artist
    artist = dsshow(
        block_df, 
        ds.Point('x', 'y'), 
        ds.mean('v'),
        cmap=fire,
        norm='eq_hist',
        ax=grid[0],
        x_range=(offset, offset + n_i),
        y_range=(offset, offset + n_i)
    )
    
    # Add colorbar
    cbar = plt.colorbar(artist, cax=grid.cbar_axes[0])
    
    data_flat = v[~np.isnan(v)]
    if len(data_flat) > 0:
        vmin, vmax = data_flat.min(), data_flat.max()
        cbar.set_ticks([vmin, 0, vmax])
        cbar.set_ticklabels([f'{vmin:.3f}', '0', f'{vmax:.3f}'])

    # Style the colorbar
    cbar.ax.tick_params(colors='white', labelsize=8)
    cbar.set_label('Value', rotation=270, labelpad=20, 
                   color='white', fontsize=10)

    # Add title
    grid[0].set_title(block_display_name, color='white', fontsize=12, pad=10)
    grid[0].set_axis_off()
    
    # Add stats text
    data_flat = v[~np.isnan(v)]
    if len(data_flat) > 0:
        stats_text = f'min={data_flat.min():.3f}\nmax={data_flat.max():.3f}\nmean={data_flat.mean():.3f}'
        grid[0].text(0.02, 0.98, stats_text, transform=grid[0].transAxes, 
                     fontsize=8, color='cyan', verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
    
    # Save the figure
    output_path = str(output_folder / f"{block_filename}_grid.png")
    plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='black')
    plt.close(fig)
    
    print(f"✓ Saved block {i} with grid style: {block_filename}_grid.png")

    # Collect for full visualization
    xs.append(x); ys.append(y); vs.append(v)
    offset += n_i

# After the loop, create the full diagonal visualization
if xs:
    X = np.concatenate(xs)
    Y = np.concatenate(ys)
    V = np.concatenate(vs)

    print("Total emitted points:", len(V))
    print("Global matrix size:", offset, "x", offset)

    df_full = pd.DataFrame({"x": X.astype(np.int64), "y": Y.astype(np.int64), "v": V.astype(np.float32)})

    # Create full diagonal visualization
    fig_full = plt.figure(figsize=(12, 12), facecolor='black')
    grid_full = ImageGrid(fig_full, 111, nrows_ncols=(1, 1), 
                          axes_pad=0.5, share_all=True,
                          cbar_location="right", cbar_mode="single", 
                          cbar_size="5%", cbar_pad="2%")
    
    artist_full = dsshow(
        df_full, 
        ds.Point('x', 'y'), 
        ds.mean('v'),
        cmap=fire,
        # shade_hows={"v": "eq_hist"},
        norm='eq_hist',
        aspect='equal',
        # span=(V.min(), V.max()),  # Explicitly set the data range
        ax=grid_full[0],
        x_range=(0, offset),
        y_range=(0, offset)
    )
    
    cbar_full = plt.colorbar(artist_full, cax=grid_full.cbar_axes[0])
    cbar_full.ax.tick_params(colors='white', labelsize=8)
    cbar_full.set_label('Attention Value', rotation=270, labelpad=20, 
                        color='white', fontsize=10)
    
    grid_full[0].set_title(f"Full Diagonal Attention (n={offset})", color='white', fontsize=14, pad=10)
    grid_full[0].set_axis_off()
    
    # Add stats for full visualization
    data_flat_full = V[~np.isnan(V)]
    if len(data_flat_full) > 0:
        stats_text_full = f'min={data_flat_full.min():.3f}\nmax={data_flat_full.max():.3f}\nmean={data_flat_full.mean():.3f}'
        grid_full[0].text(0.02, 0.98, stats_text_full, transform=grid_full[0].transAxes, 
                          fontsize=10, color='cyan', verticalalignment='top',
                          bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
    
    full_output_path = str(output_folder / "full_diagonal_grid.png")
    plt.savefig(full_output_path, dpi=150, bbox_inches='tight', facecolor='black')
    plt.close(fig_full)
    
    print(f"✓ Saved full diagonal: full_diagonal_grid.png")
else:
    print("No blocks to visualize.")


In [ ]:
ckpt_path = "out_adagramps_bs512_lr1e-2_rank5/5000_ckpt.pt"

print(f"Loading nanoGPT checkpoint from {ckpt_path}...")
checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
# Build model from model_args (nanoGPT style)
config = GPTConfig(**checkpoint["model_args"])
if hasattr(config, "flash"):
    config.flash = False
model = GPT(config)
# Strip _orig_mod. prefix (torch.compile artifact)
sd = checkpoint["model"] if "model" in checkpoint else checkpoint
sd = {k.replace("_orig_mod.", ""): v for k, v in sd.items()}
# strict=False ignores extra keys like attn.bias
model.load_state_dict(sd, strict=False)

# Print ALL layer names + shapes (put anywhere after model.load_state_dict(...))
# Fixed version (list(param.shape) can't be formatted with :20s)
print("=== ALL MODEL PARAMETER NAMES ===")
for name, param in model.named_parameters():
    shape_str = str(param.shape)
    print(f"{name:50s} {shape_str:20s} {param.numel():10,d}")

print("\n=== FIRST BLOCK'S PARAMETER NAMES ===")
first_block = model.transformer.h[0]
for name, param in first_block.named_parameters():
    shape_str = str(param.shape)
    print(f"{name:40s} {shape_str:15s} {param.numel():10,d}")


print("\n=== ATTENTION BLOCKS ONLY (blocks 0-2) ===")
# for i in range(3):
# print(f"\n--- Block {0} attn params ---")
for name, param in model.transformer.h[0].named_parameters():
    if "attn" in name:
        print(f"{name:40s} {shape_str:15s} {param.numel():10,d}")

In [1]:
import os
import gc
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
import tiktoken

from model import GPT, GPTConfig

# ── config ────────────────────────────────────────────────────────────────────
CKPT_DIR       = "checkpoints/out_adamw_bs16_lr1e-4"
CKPT_STEPS     = list(range(250, 5001, 250))
PLOT_DIR       = "plots_weights_grads_adamw"
NUM_LAYERS     = 6
NUM_EXAMPLES   = 50
BLOCK_SIZE_CAP = 128
STEP_WIDTH     = len(str(max(CKPT_STEPS)))

os.makedirs(PLOT_DIR, exist_ok=True)
device = torch.device("cpu")

PART_NAMES = ["ln_1", "attn.c_attn", "attn.c_proj", "ln_2", "mlp.c_fc", "mlp.c_proj"]

# ── helpers ───────────────────────────────────────────────────────────────────
def iter_parts(block):
    return [
        ("ln_1",        block.ln_1),
        ("attn.c_attn", block.attn.c_attn),
        ("attn.c_proj", block.attn.c_proj),
        ("ln_2",        block.ln_2),
        ("mlp.c_fc",    block.mlp.c_fc),
        ("mlp.c_proj",  block.mlp.c_proj),
    ]

def get_weight(module):
    if not (hasattr(module, "weight") and module.weight is not None):
        return None
    w = module.weight.detach().float()
    return w.unsqueeze(1) if w.dim() == 1 else w

def get_grad(module):
    """Same shape as weight but from .grad — averaged over examples."""
    if not (hasattr(module, "weight") and module.weight is not None):
        return None
    g = module.weight.grad
    if g is None:
        return None
    g = g.detach().float()
    return g.unsqueeze(1) if g.dim() == 1 else g

def make_xy_from_text(text, vocab_size, block_size, enc):
    ids = enc.encode(text)
    ids = [min(max(int(t), 0), vocab_size - 1) for t in ids]
    ids = ids[: block_size + 1]
    if len(ids) < 2:
        ids = ids + [0]
    x = torch.tensor(ids[:-1], dtype=torch.long)[None, :]
    y = torch.tensor(ids[1:],  dtype=torch.long)[None, :]
    return x, y

def savefig(path):
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[saved] {path}")

def plot_block_panel(W_dict, G_dict, bi, step, tag):
    """
    Two rows per block: top = weights, bottom = gradients.
    Columns = parts. Proportional widths preserved.
    """
    parts = [(pn, W_dict.get((bi, pn)), G_dict.get((bi, pn))) for pn in PART_NAMES]
    parts = [(pn, W, G) for pn, W, G in parts if W is not None]
    if not parts:
        return

    FIXED_H = 3.5
    widths  = [max(0.5, W.shape[1] / W.shape[0]) * FIXED_H for _, W, _ in parts]

    fig, axes = plt.subplots(
        2, len(parts),
        figsize=(sum(widths) + len(parts) * 0.4, FIXED_H * 2 + 1.5),
        gridspec_kw={"width_ratios": widths}
    )
    # ensure axes is always 2D
    if len(parts) == 1:
        axes = axes.reshape(2, 1)

    fig.suptitle(f"Weights (top) & Gradients (bottom)  —  block {bi},  step {step}", fontsize=11)

    for col, (pname, W, G) in enumerate(parts):
        for row, (mat, row_label) in enumerate([(W, "weight"), (G, "gradient")]):
            ax = axes[row, col]
            if mat is None:
                ax.axis("off")
                ax.set_title(f"{pname}\n(no grad)", fontsize=8)
                continue

            M    = mat.numpy()
            vmax = float(np.abs(M).max()) or 1.0

            im = ax.imshow(
                M,
                cmap="RdBu_r",
                vmin=-vmax, vmax=vmax,
                aspect="equal",
                interpolation="nearest"
            )
            if row == 0:
                ax.set_title(f"{pname}\n{M.shape[0]}×{M.shape[1]}", fontsize=8)
            ax.set_ylabel(row_label, fontsize=7)
            ax.tick_params(labelsize=5)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, f"weights_grads_block{bi}_step{tag}.png")
    savefig(path)

# ── dataset ───────────────────────────────────────────────────────────────────
print("Loading dataset...")
with open("data/shakespeare_char/input.txt", "r") as f:
    raw_text = f.read()

chunk_size = 512
chunks     = [raw_text[i:i + chunk_size] for i in range(0, len(raw_text), chunk_size)]
chunks     = chunks[:NUM_EXAMPLES]
enc        = tiktoken.get_encoding("gpt2")

# ── main loop ─────────────────────────────────────────────────────────────────
for step in CKPT_STEPS:
    tag       = str(step).zfill(STEP_WIDTH)
    ckpt_path = os.path.join(CKPT_DIR, f"{step}_ckpt.pt")

    if not os.path.exists(ckpt_path):
        print(f"[skip] {ckpt_path}")
        continue

    print(f"\n{'='*50}\n  step={step}\n{'='*50}")

    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    config = GPTConfig(**checkpoint["model_args"])
    if hasattr(config, "flash"):
        config.flash = False

    model = GPT(config)
    sd = checkpoint.get("model", checkpoint)
    sd = {k.replace("_orig_mod.", ""): v for k, v in sd.items()}
    model.load_state_dict(sd, strict=False)
    model.to(device).train()   # train mode so grads flow

    vocab_size       = int(checkpoint["model_args"]["vocab_size"])
    model_block_size = int(checkpoint["model_args"]["block_size"])
    block_size       = min(BLOCK_SIZE_CAP, model_block_size)

    transformer_blocks = model.transformer.h[:NUM_LAYERS]

    # ── extract weights before backward ──────────────────────────────────────
    W_dict = {}
    for bi, block in enumerate(transformer_blocks):
        for pname, pmod in iter_parts(block):
            W = get_weight(pmod)
            if W is not None:
                W_dict[(bi, pname)] = W.clone()

    # ── accumulate gradients over examples ───────────────────────────────────
    model.zero_grad(set_to_none=True)

    for idx, chunk in enumerate(tqdm(chunks, desc=f"Grads step={step}")):
        x, y = make_xy_from_text(chunk, vocab_size, block_size, enc)
        x, y = x.to(device), y.to(device)
        _, loss = model(x, targets=y)
        (loss / NUM_EXAMPLES).backward()   # accumulate mean gradient

    # ── extract accumulated gradients ─────────────────────────────────────────
    G_dict = {}
    for bi, block in enumerate(transformer_blocks):
        for pname, pmod in iter_parts(block):
            G = get_grad(pmod)
            if G is not None:
                G_dict[(bi, pname)] = G.clone()

    del model
    gc.collect()

    # ── plot ──────────────────────────────────────────────────────────────────
    for bi in range(NUM_LAYERS):
        plot_block_panel(W_dict, G_dict, bi, step, tag)

    del W_dict, G_dict
    gc.collect()

print("\nDone!")

Loading dataset...

  step=250
number of parameters: 10.65M


Grads step=250: 100%|██████████| 50/50 [41:02<00:00, 49.26s/it]


[saved] plots_weights_grads_adamw/weights_grads_block0_step0250.png
[saved] plots_weights_grads_adamw/weights_grads_block1_step0250.png
[saved] plots_weights_grads_adamw/weights_grads_block2_step0250.png
[saved] plots_weights_grads_adamw/weights_grads_block3_step0250.png
[saved] plots_weights_grads_adamw/weights_grads_block4_step0250.png
[saved] plots_weights_grads_adamw/weights_grads_block5_step0250.png

  step=500
number of parameters: 10.65M


Grads step=500:   2%|▏         | 1/50 [01:26<1:11:02, 86.99s/it]


KeyboardInterrupt: 

In [ ]:

import os
import gc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
from tqdm import tqdm
import tiktoken

from model import GPT, GPTConfig


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

CKPT_DIR        = "out_adamw"
CKPT_STEPS      = list(range(250, 5001, 250))
PLOT_DIR        = "hessians_adamW_layers"
NUM_LAYERS      = 6
NUM_EXAMPLES    = 50
BLOCK_SIZE_CAP  = 128
STEP_WIDTH      = len(str(max(CKPT_STEPS)))

COMPUTE_HESSIAN  = True
# Number of batches to average the Hessian over.
# More = lower noise, but each batch adds N extra backward passes.
# For these layer sizes (max N=3600) even 1-2 batches is fairly clean.
HESSIAN_BATCHES  = 2

device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PART_NAMES = ["ln_1", "attn.c_attn", "attn.c_proj", "ln_2", "mlp.c_fc", "mlp.c_proj"]

# Layers to skip for full Hessian (embedding tables — large N, less interesting structure)
# Set to [] to compute for all layers including wte/wpe
HESSIAN_SKIP_PARTS = []   # e.g. ["ln_1", "ln_2"] to skip LayerNorm


# ══════════════════════════════════════════════════════════════════════════════
# DIRECTORY HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def step_dir(tag):
    d = os.path.join(PLOT_DIR, f"step_{tag}")
    os.makedirs(d, exist_ok=True)
    return d

def part_dir(tag, pname):
    d = os.path.join(step_dir(tag), pname)
    os.makedirs(d, exist_ok=True)
    return d

def wg_dir(tag):
    d = os.path.join(step_dir(tag), "weights_grads")
    os.makedirs(d, exist_ok=True)
    return d


# ══════════════════════════════════════════════════════════════════════════════
# HELPERS — shared
# ══════════════════════════════════════════════════════════════════════════════

def load_checkpoint(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    config = GPTConfig(**checkpoint["model_args"])
    if hasattr(config, "flash"):
        config.flash = False          # flash attn is incompatible with double backward
    model = GPT(config)
    sd = checkpoint.get("model", checkpoint)
    sd = {k.replace("_orig_mod.", ""): v for k, v in sd.items()}
    result = model.load_state_dict(sd, strict=False)
    if result.missing_keys:
        print("  Missing   :", result.missing_keys)
    if result.unexpected_keys:
        print("  Unexpected:", result.unexpected_keys)
    return model, checkpoint


def make_xy_from_text(text, vocab_size, block_size, enc):
    ids = enc.encode(text)
    ids = [min(max(int(t), 0), vocab_size - 1) for t in ids]
    ids = ids[: block_size + 1]
    if len(ids) < 2:
        ids = ids + [0]
    x = torch.tensor(ids[:-1], dtype=torch.long)[None, :]
    y = torch.tensor(ids[1:],  dtype=torch.long)[None, :]
    return x, y


def iter_parts(block):
    return [
        ("ln_1",        block.ln_1),
        ("attn.c_attn", block.attn.c_attn),
        ("attn.c_proj", block.attn.c_proj),
        ("ln_2",        block.ln_2),
        ("mlp.c_fc",    block.mlp.c_fc),
        ("mlp.c_proj",  block.mlp.c_proj),
    ]


def get_weight(module):
    if not (hasattr(module, "weight") and module.weight is not None):
        return None
    w = module.weight.detach().float()
    return w.unsqueeze(1) if w.dim() == 1 else w


def get_grad(module):
    if not (hasattr(module, "weight") and module.weight is not None):
        return None
    g = module.weight.grad
    if g is None:
        return None
    g = g.detach().float()
    return g.unsqueeze(1) if g.dim() == 1 else g


def savefig(path):
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [saved] {path}")


def _sdp_context():
    """Use math SDPA (not flash) so that double backward works on CUDA."""
    if device.type == "cuda":
        return torch.backends.cuda.sdp_kernel(
            enable_flash=False, enable_mem_efficient=False, enable_math=True
        )
    import contextlib
    return contextlib.nullcontext()


# ══════════════════════════════════════════════════════════════════════════════
# WEIGHTS / GRADIENTS
# ══════════════════════════════════════════════════════════════════════════════

def plot_block_panel(W_dict, G_dict, bi, step, tag):
    parts = [(pn, W_dict.get((bi, pn)), G_dict.get((bi, pn))) for pn in PART_NAMES]
    parts = [(pn, W, G) for pn, W, G in parts if W is not None]
    if not parts:
        return
    FIXED_H = 3.5
    widths  = [max(0.5, W.shape[1] / W.shape[0]) * FIXED_H for _, W, _ in parts]
    fig, axes = plt.subplots(
        2, len(parts),
        figsize=(sum(widths) + len(parts) * 0.4, FIXED_H * 2 + 1.5),
        gridspec_kw={"width_ratios": widths},
    )
    if len(parts) == 1:
        axes = axes.reshape(2, 1)
    fig.suptitle(f"Weights (top) & Gradients (bottom)  —  block {bi},  step {step}", fontsize=11)
    for col, (pname, W, G) in enumerate(parts):
        for row, (mat, row_label) in enumerate([(W, "weight"), (G, "gradient")]):
            ax = axes[row, col]
            if mat is None:
                ax.axis("off"); ax.set_title(f"{pname}\n(no grad)", fontsize=8); continue
            M    = mat.cpu().numpy()
            vmax = float(np.abs(M).max()) or 1.0
            im   = ax.imshow(M, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                             aspect="equal", interpolation="nearest")
            if row == 0:
                ax.set_title(f"{pname}\n{M.shape[0]}×{M.shape[1]}", fontsize=8)
            ax.set_ylabel(row_label, fontsize=7)
            ax.tick_params(labelsize=5)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    savefig(os.path.join(wg_dir(tag), f"block{bi}.png"))


def plot_part_wg(W, G, pname, bi, step, tag):
    rows = [(m, l) for m, l in [(W, "weight"), (G, "gradient")] if m is not None]
    if not rows:
        return
    fig, axes = plt.subplots(1, len(rows), figsize=(4 * len(rows), 3.5))
    if len(rows) == 1:
        axes = [axes]
    fig.suptitle(f"{pname}  —  block {bi},  step {step}", fontsize=10)
    for ax, (mat, label) in zip(axes, rows):
        M    = mat.cpu().numpy()
        vmax = float(np.abs(M).max()) or 1.0
        im   = ax.imshow(M, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                         aspect="equal", interpolation="nearest")
        ax.set_title(label, fontsize=9)
        ax.tick_params(labelsize=5)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    savefig(os.path.join(part_dir(tag, pname), f"weights_grads_block{bi}.png"))


# ══════════════════════════════════════════════════════════════════════════════
# FULL HESSIAN COMPUTATION
# ══════════════════════════════════════════════════════════════════════════════

def compute_full_hessian(model, param, xs, ys):
    N = param.numel()
    total_H = np.zeros((N, N), dtype=np.float64)

    for x, y in zip(xs, ys):
        model.zero_grad(set_to_none=True)

        with _sdp_context():          # ✅ fresh context manager each iteration
            _, loss = model(x.to(device), targets=y.to(device))

        (grad_param,) = torch.autograd.grad(loss, param, create_graph=True)
        grad_flat = grad_param.flatten()

        H_batch = torch.zeros(N, N)
        for j in range(N):
            (row,) = torch.autograd.grad(
                grad_flat[j], param,
                retain_graph=(j < N - 1),
            )
            H_batch[j] = row.flatten().detach().cpu()

        total_H += H_batch.numpy().astype(np.float64)
        del grad_param, grad_flat, H_batch

    H = total_H / len(xs)
    return (H + H.T) / 2


# ══════════════════════════════════════════════════════════════════════════════
# FULL HESSIAN PLOTS
# ══════════════════════════════════════════════════════════════════════════════

def _draw_neuron_grid(ax, d_out, d_in):
    """Overlay a light grid at neuron (output-neuron) boundaries."""
    if d_in <= 1:
        return
    N = d_out * d_in
    # Grid lines at every d_in interval in flat-parameter space
    for k in range(1, d_out):
        pos = k * d_in - 0.5
        ax.axhline(pos, color="white", lw=0.3, alpha=0.4)
        ax.axvline(pos, color="white", lw=0.3, alpha=0.4)


def plot_full_hessian(H, param_shape, pname, bi, step, tag):
    N = H.shape[0]
    if len(param_shape) == 1:
        d_out, d_in = param_shape[0], 1
    else:
        d_out, d_in = param_shape[0], param_shape[1]

    Habs = np.abs(H)
    vmax = np.percentile(Habs, 99) or 1.0

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    im0 = axes[0].imshow(H, cmap="RdBu_r", aspect="auto", origin="upper",
                         vmin=-vmax, vmax=vmax, interpolation="nearest")
    _draw_neuron_grid(axes[0], d_out, d_in)
    axes[0].set_title("Linear (RdBu, p99 clip)", fontsize=9)
    axes[0].set_xlabel("param index")
    axes[0].set_ylabel("param index")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(np.log1p(Habs), cmap="viridis", aspect="auto",
                         origin="upper", interpolation="nearest")
    _draw_neuron_grid(axes[1], d_out, d_in)
    axes[1].set_title("|H|, log1p scale", fontsize=9)
    axes[1].set_xlabel("param index")
    axes[1].set_ylabel("param index")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    fig.suptitle(
        f"Full Hessian  —  {pname}  ·  block {bi}  ·  step {step}\n"
        f"shape {d_out}×{d_in}  |  N={N}",
        fontsize=10,
    )
    plt.tight_layout()
    savefig(os.path.join(part_dir(tag, pname), f"full_hessian_block{bi}.png"))

def plot_neuron_hessian(H, param_shape, pname, bi, step, tag):
    """
    Neuron-level (d_out × d_out) Hessian: average |H| within each (d_in × d_in) block.
    This collapses the full N×N Hessian into a d_out×d_out coupling matrix.
    If the theory holds, this should be near-diagonal (neurons decouple).
    Saved alongside the full Hessian for direct comparison.
    """
    if len(param_shape) == 1:
        return                      # scalar param — no neuron structure
    d_out, d_in = param_shape[0], param_shape[1]
    N = d_out * d_in
    H_trim = H[:N, :N]

    # Reshape to (d_out, d_in, d_out, d_in) and mean over the d_in axes
    # → (d_out, d_out) neuron coupling matrix
    H_neuron = (np.abs(H_trim)
                .reshape(d_out, d_in, d_out, d_in)
                .mean(axis=(1, 3)))

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    im0 = axes[0].imshow(H_neuron, cmap="viridis", aspect="auto", origin="upper")
    axes[0].set_title("Linear scale")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(np.log1p(H_neuron), cmap="viridis", aspect="auto", origin="upper")
    axes[1].set_title("Log1p scale")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    for ax in axes:
        ax.set_xlabel(f"output neuron (d_out={d_out})")
        ax.set_ylabel(f"output neuron (d_out={d_out})")

    fig.suptitle(
        f"Neuron-level Hessian  —  {pname}  ·  block {bi}  ·  step {step}\n"
        f"weight {d_out}×{d_in}, compressed to {d_out}×{d_out}",
        fontsize=9,
    )
    savefig(os.path.join(part_dir(tag, pname), f"neuron_hessian_block{bi}.png"))


# ══════════════════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════════════════

print("Loading dataset…")
with open("data/shakespeare_char/input.txt", "r") as f:
    raw_text = f.read()

chunk_size = 512
chunks     = [raw_text[i:i + chunk_size] for i in range(0, len(raw_text), chunk_size)]
enc        = tiktoken.get_encoding("gpt2")

grad_chunks    = chunks[:NUM_EXAMPLES]
hessian_chunks = chunks[NUM_EXAMPLES: NUM_EXAMPLES + HESSIAN_BATCHES]


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

for step in CKPT_STEPS:
    tag       = str(step).zfill(STEP_WIDTH)
    ckpt_path = os.path.join(CKPT_DIR, f"{step}_ckpt.pt")

    if not os.path.exists(ckpt_path):
        print(f"[skip] {ckpt_path}")
        continue

    print(f"\n{'='*55}\n  step = {step}\n{'='*55}")

    model, checkpoint = load_checkpoint(ckpt_path)
    vocab_size  = int(checkpoint["model_args"]["vocab_size"])
    block_size  = min(BLOCK_SIZE_CAP, int(checkpoint["model_args"]["block_size"]))

    model.to(device).train()
    transformer_blocks = model.transformer.h[:NUM_LAYERS]

    # ── weights ───────────────────────────────────────────────────────────────
    W_dict = {}
    for bi, block in enumerate(transformer_blocks):
        for pname, pmod in iter_parts(block):
            W = get_weight(pmod)
            if W is not None:
                W_dict[(bi, pname)] = W.clone()

    # ── gradients ─────────────────────────────────────────────────────────────
    model.zero_grad(set_to_none=True)
    for chunk in tqdm(grad_chunks, desc=f"Grads step={step}"):
        x, y = make_xy_from_text(chunk, vocab_size, block_size, enc)
        _, loss = model(x.to(device), targets=y.to(device))
        (loss / NUM_EXAMPLES).backward()

    G_dict = {}
    for bi, block in enumerate(transformer_blocks):
        for pname, pmod in iter_parts(block):
            G = get_grad(pmod)
            if G is not None:
                G_dict[(bi, pname)] = G.clone()

    # ── weight+gradient plots ─────────────────────────────────────────────────
    for bi in range(NUM_LAYERS):
        plot_block_panel(W_dict, G_dict, bi, step, tag)
        for pname in PART_NAMES:
            W = W_dict.get((bi, pname))
            G = G_dict.get((bi, pname))
            if W is not None:
                plot_part_wg(W, G, pname, bi, step, tag)
    del W_dict, G_dict

    # ── full hessian ──────────────────────────────────────────────────────────
    if COMPUTE_HESSIAN:
        hess_xy = [
            make_xy_from_text(c, vocab_size, block_size, enc)
            for c in hessian_chunks
        ]
        hess_xs = [xy[0] for xy in hess_xy]
        hess_ys = [xy[1] for xy in hess_xy]

        for bi, block in enumerate(tqdm(transformer_blocks, desc=f"Hessian step={step}")):
            model.zero_grad(set_to_none=True)
            for pname, pmod in iter_parts(block):
                if pname in HESSIAN_SKIP_PARTS:
                    continue
                if not (hasattr(pmod, "weight") and pmod.weight is not None):
                    continue

                param       = pmod.weight
                param_shape = tuple(param.shape)
                N           = param.numel()

                print(f"  Full Hessian block{bi}/{pname}  shape={param_shape}  N={N}")

                try:
                    H = compute_full_hessian(model, param, hess_xs, hess_ys)

                    # 1. Full matrix + spectrum plot
                    plot_full_hessian(H, param_shape, pname, bi, step, tag)

                    # # 2. Neuron-level compressed view (d_out × d_out)
                    # plot_neuron_hessian(H, param_shape, pname, bi, step, tag)

                    # 3. Save raw Hessian as .npy for later analysis
                    # npy_path = os.path.join(
                    #     part_dir(tag, pname), f"hessian_block{bi}.npy"
                    # )
                    # np.save(npy_path, H)
                    # print(f"    saved .npy: {npy_path}")

                except Exception as e:
                    print(f"  [warn] block{bi}/{pname} skipped: {e}")

        del hess_xs, hess_ys, hess_xy

    # ── cleanup ───────────────────────────────────────────────────────────────
    del model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("\nDone!")


Loading dataset…
[skip] out_adamw/250_ckpt.pt
[skip] out_adamw/500_ckpt.pt
[skip] out_adamw/750_ckpt.pt
[skip] out_adamw/1000_ckpt.pt
[skip] out_adamw/1250_ckpt.pt
[skip] out_adamw/1500_ckpt.pt
[skip] out_adamw/1750_ckpt.pt
[skip] out_adamw/2000_ckpt.pt
[skip] out_adamw/2250_ckpt.pt
[skip] out_adamw/2500_ckpt.pt
[skip] out_adamw/2750_ckpt.pt
[skip] out_adamw/3000_ckpt.pt
[skip] out_adamw/3250_ckpt.pt
[skip] out_adamw/3500_ckpt.pt
[skip] out_adamw/3750_ckpt.pt
[skip] out_adamw/4000_ckpt.pt
[skip] out_adamw/4250_ckpt.pt
[skip] out_adamw/4500_ckpt.pt
[skip] out_adamw/4750_ckpt.pt

  step = 5000
number of parameters: 0.07M


Grads step=5000:   0%|          | 0/50 [00:00<?, ?it/s]

Grads step=5000: 100%|██████████| 50/50 [00:00<00:00, 114.54it/s]


  [saved] hessians_adamW_layers/step_5000/weights_grads/block0.png
  [saved] hessians_adamW_layers/step_5000/ln_1/weights_grads_block0.png
  [saved] hessians_adamW_layers/step_5000/attn.c_attn/weights_grads_block0.png
  [saved] hessians_adamW_layers/step_5000/attn.c_proj/weights_grads_block0.png
  [saved] hessians_adamW_layers/step_5000/ln_2/weights_grads_block0.png
  [saved] hessians_adamW_layers/step_5000/mlp.c_fc/weights_grads_block0.png
  [saved] hessians_adamW_layers/step_5000/mlp.c_proj/weights_grads_block0.png
  [saved] hessians_adamW_layers/step_5000/weights_grads/block1.png
  [saved] hessians_adamW_layers/step_5000/ln_1/weights_grads_block1.png
  [saved] hessians_adamW_layers/step_5000/attn.c_attn/weights_grads_block1.png
  [saved] hessians_adamW_layers/step_5000/attn.c_proj/weights_grads_block1.png
  [saved] hessians_adamW_layers/step_5000/ln_2/weights_grads_block1.png
  [saved] hessians_adamW_layers/step_5000/mlp.c_fc/weights_grads_block1.png
  [saved] hessians_adamW_layers/

Hessian step=5000:   0%|          | 0/6 [00:00<?, ?it/s]

  Full Hessian block0/ln_1  shape=(30,)  N=30
  [saved] hessians_adamW_layers/step_5000/ln_1/full_hessian_block0.png
  Full Hessian block0/attn.c_attn  shape=(90, 30)  N=2700
  [saved] hessians_adamW_layers/step_5000/attn.c_attn/full_hessian_block0.png
  Full Hessian block0/attn.c_proj  shape=(30, 30)  N=900
  [saved] hessians_adamW_layers/step_5000/attn.c_proj/full_hessian_block0.png
  Full Hessian block0/ln_2  shape=(30,)  N=30
  [saved] hessians_adamW_layers/step_5000/ln_2/full_hessian_block0.png
  Full Hessian block0/mlp.c_fc  shape=(120, 30)  N=3600
  [saved] hessians_adamW_layers/step_5000/mlp.c_fc/full_hessian_block0.png
  Full Hessian block0/mlp.c_proj  shape=(30, 120)  N=3600


Hessian step=5000:  17%|█▋        | 1/6 [05:32<27:40, 332.02s/it]

  [saved] hessians_adamW_layers/step_5000/mlp.c_proj/full_hessian_block0.png
  Full Hessian block1/ln_1  shape=(30,)  N=30
  [saved] hessians_adamW_layers/step_5000/ln_1/full_hessian_block1.png
  Full Hessian block1/attn.c_attn  shape=(90, 30)  N=2700
  [saved] hessians_adamW_layers/step_5000/attn.c_attn/full_hessian_block1.png
  Full Hessian block1/attn.c_proj  shape=(30, 30)  N=900
  [saved] hessians_adamW_layers/step_5000/attn.c_proj/full_hessian_block1.png
  Full Hessian block1/ln_2  shape=(30,)  N=30
  [saved] hessians_adamW_layers/step_5000/ln_2/full_hessian_block1.png
  Full Hessian block1/mlp.c_fc  shape=(120, 30)  N=3600


In [1]:
import os
import gc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from tqdm import tqdm
import tiktoken

from model import GPT, GPTConfig


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

CKPT_DIR        = "out_adamw"
CKPT_STEPS      = list(range(250, 5001, 250))
PLOT_DIR        = "hessians_adamW_layers"
NUM_LAYERS      = 6
NUM_EXAMPLES    = 50
BLOCK_SIZE_CAP  = 128
STEP_WIDTH      = len(str(max(CKPT_STEPS)))
N_HEAD          = 6

COMPUTE_HESSIAN  = True
HESSIAN_BATCHES  = 2

# Tikhonov damping: λ = DAMPING * ||H||_F / N
INV_HESSIAN_DAMPING = 1e-3

device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PART_NAMES = ["ln_1", "attn.c_attn", "attn.c_proj", "ln_2", "mlp.c_fc", "mlp.c_proj"]
HESSIAN_SKIP_PARTS = []

_QKV_COLORS = {"Q": "#4878CF", "K": "#6ACC65", "V": "#D65F5F"}
_QKV_NAMES  = ["Q", "K", "V"]


# ══════════════════════════════════════════════════════════════════════════════
# DIRECTORY HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def step_dir(tag):
    d = os.path.join(PLOT_DIR, f"step_{tag}")
    os.makedirs(d, exist_ok=True)
    return d

def part_dir(tag, pname):
    d = os.path.join(step_dir(tag), pname)
    os.makedirs(d, exist_ok=True)
    return d


# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def load_checkpoint(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    config = GPTConfig(**checkpoint["model_args"])
    if hasattr(config, "flash"):
        config.flash = False
    model = GPT(config)
    sd = checkpoint.get("model", checkpoint)
    sd = {k.replace("_orig_mod.", ""): v for k, v in sd.items()}
    result = model.load_state_dict(sd, strict=False)
    if result.missing_keys:
        print("  Missing   :", result.missing_keys)
    if result.unexpected_keys:
        print("  Unexpected:", result.unexpected_keys)
    return model, checkpoint

def make_xy_from_text(text, vocab_size, block_size, enc):
    ids = enc.encode(text)
    ids = [min(max(int(t), 0), vocab_size - 1) for t in ids]
    ids = ids[: block_size + 1]
    if len(ids) < 2:
        ids = ids + [0]
    x = torch.tensor(ids[:-1], dtype=torch.long)[None, :]
    y = torch.tensor(ids[1:],  dtype=torch.long)[None, :]
    return x, y

def iter_parts(block):
    return [
        ("ln_1",        block.ln_1),
        ("attn.c_attn", block.attn.c_attn),
        ("attn.c_proj", block.attn.c_proj),
        ("ln_2",        block.ln_2),
        ("mlp.c_fc",    block.mlp.c_fc),
        ("mlp.c_proj",  block.mlp.c_proj),
    ]

def savefig(path):
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [saved] {path}")

def _sdp_context():
    if device.type == "cuda":
        return torch.backends.cuda.sdp_kernel(
            enable_flash=False, enable_mem_efficient=False, enable_math=True
        )
    import contextlib
    return contextlib.nullcontext()


# ══════════════════════════════════════════════════════════════════════════════
# HESSIAN COMPUTATION
# ══════════════════════════════════════════════════════════════════════════════

def compute_fisher_from_grads(grad_list: list[np.ndarray]) -> np.ndarray:
    """
    Empirical Fisher from a list of pre-computed flat gradient vectors:

        F = (1/B) Σ_b  g_b g_b^T

    grad_list  —  list of (N,) float64 numpy arrays, one per sample.
    No model access needed; pure numpy.
    """
    N       = grad_list[0].shape[0]
    total_F = np.zeros((N, N), dtype=np.float64)
    for gv in grad_list:
        total_F += np.outer(gv, gv)
    F = total_F / len(grad_list)
    return (F + F.T) / 2

def compute_inv_fisher(F: np.ndarray, damping: float = INV_HESSIAN_DAMPING) -> np.ndarray:
    """
    Regularised inverse Fisher (natural-gradient preconditioner):

        F_inv = (F + λ I)^{-1},   λ = damping * ||F||_F / N

    This is the preconditioner used by natural gradient / K-FAC methods.
    """
    N   = F.shape[0]
    F_reg = F
    F_inv = np.linalg.inv(F_reg)
    return (F_inv + F_inv.T) / 2

def compute_full_hessian(model, param, xs, ys):
    N        = param.numel()
    total_H  = np.zeros((N, N), dtype=np.float64)
    grads    = []                          # ← collected for Fisher reuse

    for x, y in zip(xs, ys):
        model.zero_grad(set_to_none=True)
        with _sdp_context():
            _, loss = model(x.to(device), targets=y.to(device))

        (grad_param,) = torch.autograd.grad(loss, param, create_graph=True)
        grad_flat = grad_param.flatten()

        # collect flat gradient for Fisher (detach before double-backward)
        grads.append(grad_flat.detach().float().cpu().numpy().astype(np.float64))

        H_batch = torch.zeros(N, N, device=device)
        for j in range(N):
            (row,) = torch.autograd.grad(
                grad_flat[j], param,
                retain_graph=(j < N - 1),
            )
            H_batch[j] = row.flatten().detach()

        total_H += H_batch.cpu().numpy().astype(np.float64)
        del grad_param, grad_flat, H_batch

    H = total_H / len(xs)
    return (H + H.T) / 2, grads           # ← now returns a tuple

def compute_inv_hessian(H: np.ndarray, damping: float = INV_HESSIAN_DAMPING) -> np.ndarray:
    """Tikhonov-regularised inverse: (H + λI)^{-1}, λ = damping * ||H||_F / N."""
    N = H.shape[0]
    lam = damping * float(np.linalg.norm(H, "fro")) / N
    H_reg = H
    H_inv = np.linalg.inv(H_reg)
    return (H_inv + H_inv.T) / 2


# ══════════════════════════════════════════════════════════════════════════════
# ANNOTATION HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _draw_neuron_grid(ax, d_out, d_in):
    if d_in <= 1:
        return
    for k in range(1, d_out):
        pos = k * d_in - 0.5
        ax.axhline(pos, color="white", lw=0.3, alpha=0.4)
        ax.axvline(pos, color="white", lw=0.3, alpha=0.4)

def _draw_head_grid_cattn(ax, n_head, n_embd):
    head_size = n_embd // n_head
    block     = head_size * n_embd
    minor, major = [], []
    for qkv in range(3):
        base = qkv * n_head * block
        for h in range(1, n_head):
            minor.append(base + h * block - 0.5)
        if qkv > 0:
            major.append(base - 0.5)
    for pos in minor:
        ax.axhline(pos, color="white",  lw=0.4, alpha=0.55)
        ax.axvline(pos, color="white",  lw=0.4, alpha=0.55)
    for pos in major:
        ax.axhline(pos, color="yellow", lw=1.2, alpha=0.85)
        ax.axvline(pos, color="yellow", lw=1.2, alpha=0.85)

def _add_qkv_annotations(ax, n_head, n_embd, tick_threshold=3600):
    head_size = n_embd // n_head
    block     = head_size * n_embd
    N         = 3 * n_head * block

    for qi, name in enumerate(_QKV_NAMES):
        s = qi * n_head * block - 0.5
        e = (qi + 1) * n_head * block - 0.5
        color = _QKV_COLORS[name]
        ax.axhspan(s, e, alpha=0.10, color=color, zorder=0)
        ax.axvspan(s, e, alpha=0.10, color=color, zorder=0)

    for qi, name in enumerate(_QKV_NAMES):
        mid = qi * n_head * block + n_head * block / 2 - 0.5
        color = _QKV_COLORS[name]
        for xy_args in [
            dict(x=N * 0.015, y=mid, ha="left",   va="center"),
            dict(x=mid,       y=N * 0.015, ha="center", va="top"),
        ]:
            ax.text(
                xy_args["x"], xy_args["y"], name,
                ha=xy_args["ha"], va=xy_args["va"],
                fontsize=11, fontweight="bold", color=color, alpha=0.85, zorder=5,
                bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                          alpha=0.55, edgecolor="none"),
            )

    if N <= tick_threshold:
        centers = [
            qi * n_head * block + h * block + block / 2 - 0.5
            for qi in range(3) for h in range(n_head)
        ]
        labels = [f"{_QKV_NAMES[qi]}_h{h}" for qi in range(3) for h in range(n_head)]
        ax.set_yticks(centers); ax.set_yticklabels(labels, fontsize=6)
        ax.set_xticks(centers); ax.set_xticklabels(labels, rotation=90, fontsize=6)
        for tick, lbl in zip(ax.get_yticklabels(), labels): tick.set_color(_QKV_COLORS[lbl[0]])
        for tick, lbl in zip(ax.get_xticklabels(), labels): tick.set_color(_QKV_COLORS[lbl[0]])
    else:
        centers = [(qi + 0.5) * n_head * block - 0.5 for qi in range(3)]
        ax.set_yticks(centers); ax.set_yticklabels(_QKV_NAMES, fontsize=9, fontweight="bold")
        ax.set_xticks(centers); ax.set_xticklabels(_QKV_NAMES, fontsize=9, fontweight="bold")
        for tick, name in zip(ax.get_yticklabels(), _QKV_NAMES): tick.set_color(_QKV_COLORS[name])
        for tick, name in zip(ax.get_xticklabels(), _QKV_NAMES): tick.set_color(_QKV_COLORS[name])


# ══════════════════════════════════════════════════════════════════════════════
# PLOT: FULL HESSIAN  (linear RdBu_r only)
# ══════════════════════════════════════════════════════════════════════════════

def _single_linear_panel(mat, title, suptitle, pname, d_out, d_in, n_head, n_embd):
    """Shared single-panel linear RdBu_r figure used by H and H⁻¹."""
    Mabs = np.abs(mat)
    vmax = np.percentile(Mabs, 99) or 1.0

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(mat, cmap="RdBu_r", aspect="auto", origin="upper",
                   vmin=-vmax, vmax=vmax, interpolation="nearest")

    if pname == "attn.c_attn":
        _draw_head_grid_cattn(ax, n_head, n_embd)
        _add_qkv_annotations(ax, n_head, n_embd)
        legend_patches = [
            mpatches.Patch(color=_QKV_COLORS[n], alpha=0.7, label=f"{n} (W_{n})")
            for n in _QKV_NAMES
        ]
        ax.legend(handles=legend_patches, loc="upper right",
                  fontsize=7, framealpha=0.8, title="projection")
    else:
        _draw_neuron_grid(ax, d_out, d_in)
        ax.set_xlabel("param index")
        ax.set_ylabel("param index")

    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(suptitle, fontsize=10)
    plt.tight_layout()
    return fig


def plot_full_fisher(F, param_shape, pname, bi, step, tag, n_head=N_HEAD):
    """
    Single-panel linear RdBu_r Fisher matrix plot.
    Saved as  full_fisher_block{bi}.png

    The Fisher is always PSD (outer-product construction), so the colormap
    will be one-sided (all non-negative) unless the finite-sample estimate
    introduces tiny negative values near zero.
    """
    N = F.shape[0]
    d_out, d_in = (param_shape[0], 1) if len(param_shape) == 1 else param_shape[:2]
    n_embd = d_in

    fig = _single_linear_panel(
        F,
        title="F  (empirical Fisher)  linear (RdBu_r, p99 clip)",
        suptitle=(
            f"Empirical Fisher Information Matrix  —  {pname}  ·  block {bi}  ·  step {step}\n"
            f"shape {d_out}×{d_in}  |  N={N}  |  F = (1/B) Σ g g^T"
        ),
        pname=pname, d_out=d_out, d_in=d_in, n_head=n_head, n_embd=n_embd,
    )
    savefig(os.path.join(part_dir(tag, pname), f"full_fisher_block{bi}.png"))


def plot_full_inv_fisher(F_inv, param_shape, pname, bi, step, tag, n_head=N_HEAD):
    """
    Single-panel linear RdBu_r inverse Fisher plot.
    Saved as  inv_fisher_full_block{bi}.png

    The inverse Fisher is the natural-gradient preconditioner: the Newton
    step under the Fisher metric.  Large diagonal entries indicate flat
    directions in the Fisher geometry (under-constrained parameters).
    """
    N = F_inv.shape[0]
    d_out, d_in = (param_shape[0], 1) if len(param_shape) == 1 else param_shape[:2]
    n_embd = d_in

    fig = _single_linear_panel(
        F_inv,
        title="F⁻¹  (natural-gradient preconditioner)  linear (RdBu_r, p99 clip)",
        suptitle=(
            f"Inverse Fisher (natural-gradient preconditioner)  —  {pname}  ·  block {bi}  ·  step {step}\n"
            f"shape {d_out}×{d_in}  |  N={N}  |  damping λ={INV_HESSIAN_DAMPING:.0e}"
        ),
        pname=pname, d_out=d_out, d_in=d_in, n_head=n_head, n_embd=n_embd,
    )
    savefig(os.path.join(part_dir(tag, pname), f"inv_fisher_full_block{bi}.png"))


def plot_full_hessian(H, param_shape, pname, bi, step, tag, n_head=N_HEAD):
    """Single-panel linear RdBu_r Hessian plot → full_hessian_block{bi}.png"""
    N = H.shape[0]
    d_out, d_in = (param_shape[0], 1) if len(param_shape) == 1 else param_shape[:2]
    n_embd = d_in

    fig = _single_linear_panel(
        H,
        title="H  linear (RdBu_r, p99 clip)",
        suptitle=(
            f"Full Hessian  —  {pname}  ·  block {bi}  ·  step {step}\n"
            f"shape {d_out}×{d_in}  |  N={N}"
        ),
        pname=pname, d_out=d_out, d_in=d_in, n_head=n_head, n_embd=n_embd,
    )
    savefig(os.path.join(part_dir(tag, pname), f"full_hessian_block{bi}.png"))


def plot_full_inv_hessian(H_inv, param_shape, pname, bi, step, tag, n_head=N_HEAD):
    """Single-panel linear RdBu_r inverse Hessian plot → inv_hessian_full_block{bi}.png"""
    N = H_inv.shape[0]
    d_out, d_in = (param_shape[0], 1) if len(param_shape) == 1 else param_shape[:2]
    n_embd = d_in

    fig = _single_linear_panel(
        H_inv,
        title="H⁻¹  linear (RdBu_r, p99 clip)",
        suptitle=(
            f"Inverse Hessian (preconditioner)  —  {pname}  ·  block {bi}  ·  step {step}\n"
            f"shape {d_out}×{d_in}  |  N={N}  |  damping λ={INV_HESSIAN_DAMPING:.0e}"
        ),
        pname=pname, d_out=d_out, d_in=d_in, n_head=n_head, n_embd=n_embd,
    )
    savefig(os.path.join(part_dir(tag, pname), f"inv_hessian_full_block{bi}.png"))


# ══════════════════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════════════════

print("Loading dataset…")
with open("data/shakespeare_char/input.txt", "r") as f:
    raw_text = f.read()

chunk_size     = 512
chunks         = [raw_text[i:i + chunk_size] for i in range(0, len(raw_text), chunk_size)]
enc            = tiktoken.get_encoding("gpt2")
grad_chunks    = chunks[:NUM_EXAMPLES]
hessian_chunks = chunks[NUM_EXAMPLES: NUM_EXAMPLES + HESSIAN_BATCHES]


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

for step in CKPT_STEPS:
    tag       = str(step).zfill(STEP_WIDTH)
    ckpt_path = os.path.join(CKPT_DIR, f"{step}_ckpt.pt")

    if not os.path.exists(ckpt_path):
        print(f"[skip] {ckpt_path}")
        continue

    print(f"\n{'='*55}\n  step = {step}\n{'='*55}")

    model, checkpoint = load_checkpoint(ckpt_path)
    vocab_size  = int(checkpoint["model_args"]["vocab_size"])
    block_size  = min(BLOCK_SIZE_CAP, int(checkpoint["model_args"]["block_size"]))

    model.to(device).train()
    transformer_blocks = model.transformer.h[:NUM_LAYERS]

    if COMPUTE_HESSIAN:
        hess_xy = [
            make_xy_from_text(c, vocab_size, block_size, enc)
            for c in hessian_chunks
        ]
        hess_xs = [xy[0] for xy in hess_xy]
        hess_ys = [xy[1] for xy in hess_xy]

        for bi, block in enumerate(tqdm(transformer_blocks, desc=f"Hessian step={step}")):
            model.zero_grad(set_to_none=True)

            for pname, pmod in iter_parts(block):
                if pname in HESSIAN_SKIP_PARTS:
                    continue
                if not (hasattr(pmod, "weight") and pmod.weight is not None):
                    continue

                param       = pmod.weight
                param_shape = tuple(param.shape)
                N           = param.numel()

                print(f"  Hessian block{bi}/{pname}  shape={param_shape}  N={N}")

                try:
                    # ── Hessian + Fisher (shared forward passes) ──────────────────────
                    H, grads = compute_full_hessian(model, param, hess_xs, hess_ys)

                    plot_full_hessian(H, param_shape, pname, bi, step, tag, n_head=N_HEAD)
                    H_inv = compute_inv_hessian(H, damping=INV_HESSIAN_DAMPING)
                    plot_full_inv_hessian(H_inv, param_shape, pname, bi, step, tag, n_head=N_HEAD)

                    # Fisher reuses gradients already computed above — zero extra backward passes
                    F = compute_fisher_from_grads(grads)
                    plot_full_fisher(F, param_shape, pname, bi, step, tag, n_head=N_HEAD)
                    F_inv = compute_inv_fisher(F, damping=INV_HESSIAN_DAMPING)
                    plot_full_inv_fisher(F_inv, param_shape, pname, bi, step, tag, n_head=N_HEAD)

                except Exception as e:
                    print(f"  [warn] block{bi}/{pname} skipped: {e}")

        del hess_xs, hess_ys, hess_xy

    del model
    gc.collect()
    
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("\nDone!")

Loading dataset…
[skip] out_adamw/250_ckpt.pt
[skip] out_adamw/500_ckpt.pt
[skip] out_adamw/750_ckpt.pt
[skip] out_adamw/1000_ckpt.pt
[skip] out_adamw/1250_ckpt.pt
[skip] out_adamw/1500_ckpt.pt
[skip] out_adamw/1750_ckpt.pt
[skip] out_adamw/2000_ckpt.pt
[skip] out_adamw/2250_ckpt.pt
[skip] out_adamw/2500_ckpt.pt
[skip] out_adamw/2750_ckpt.pt
[skip] out_adamw/3000_ckpt.pt
[skip] out_adamw/3250_ckpt.pt
[skip] out_adamw/3500_ckpt.pt
[skip] out_adamw/3750_ckpt.pt
[skip] out_adamw/4000_ckpt.pt
[skip] out_adamw/4250_ckpt.pt
[skip] out_adamw/4500_ckpt.pt
[skip] out_adamw/4750_ckpt.pt

  step = 5000
number of parameters: 0.07M


Hessian step=5000:   0%|          | 0/6 [00:00<?, ?it/s]/home/user/conda/lib/python3.11/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


  Hessian block0/ln_1  shape=(30,)  N=30
  [saved] hessians_adamW_layers/step_5000/ln_1/full_hessian_block0.png
  [saved] hessians_adamW_layers/step_5000/ln_1/inv_hessian_full_block0.png
  [saved] hessians_adamW_layers/step_5000/ln_1/full_fisher_block0.png
  [saved] hessians_adamW_layers/step_5000/ln_1/inv_fisher_full_block0.png
  Hessian block0/attn.c_attn  shape=(90, 30)  N=2700
  [saved] hessians_adamW_layers/step_5000/attn.c_attn/full_hessian_block0.png
  [saved] hessians_adamW_layers/step_5000/attn.c_attn/inv_hessian_full_block0.png
  [saved] hessians_adamW_layers/step_5000/attn.c_attn/full_fisher_block0.png
  [saved] hessians_adamW_layers/step_5000/attn.c_attn/inv_fisher_full_block0.png
  Hessian block0/attn.c_proj  shape=(30, 30)  N=900


/home/user/conda/lib/python3.11/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


  [saved] hessians_adamW_layers/step_5000/attn.c_proj/full_hessian_block0.png
  [saved] hessians_adamW_layers/step_5000/attn.c_proj/inv_hessian_full_block0.png
  [saved] hessians_adamW_layers/step_5000/attn.c_proj/full_fisher_block0.png
  [saved] hessians_adamW_layers/step_5000/attn.c_proj/inv_fisher_full_block0.png
  Hessian block0/ln_2  shape=(30,)  N=30
  [saved] hessians_adamW_layers/step_5000/ln_2/full_hessian_block0.png
  [saved] hessians_adamW_layers/step_5000/ln_2/inv_hessian_full_block0.png
  [saved] hessians_adamW_layers/step_5000/ln_2/full_fisher_block0.png
  [saved] hessians_adamW_layers/step_5000/ln_2/inv_fisher_full_block0.png
  Hessian block0/mlp.c_fc  shape=(120, 30)  N=3600
  [saved] hessians_adamW_layers/step_5000/mlp.c_fc/full_hessian_block0.png
  [saved] hessians_adamW_layers/step_5000/mlp.c_fc/inv_hessian_full_block0.png
  [saved] hessians_adamW_layers/step_5000/mlp.c_fc/full_fisher_block0.png
  [saved] hessians_adamW_layers/step_5000/mlp.c_fc/inv_fisher_full_block

...

/home/user/conda/lib/python3.11/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


  [saved] hessians_adamW_layers/step_5000/attn.c_proj/full_hessian_block5.png
  [saved] hessians_adamW_layers/step_5000/attn.c_proj/inv_hessian_full_block5.png
  [saved] hessians_adamW_layers/step_5000/attn.c_proj/full_fisher_block5.png
  [saved] hessians_adamW_layers/step_5000/attn.c_proj/inv_fisher_full_block5.png
  Hessian block5/ln_2  shape=(30,)  N=30
  [saved] hessians_adamW_layers/step_5000/ln_2/full_hessian_block5.png
  [saved] hessians_adamW_layers/step_5000/ln_2/inv_hessian_full_block5.png
  [saved] hessians_adamW_layers/step_5000/ln_2/full_fisher_block5.png
  [warn] block5/ln_2 skipped: Singular matrix
  Hessian block5/mlp.c_fc  shape=(120, 30)  N=3600
  [saved] hessians_adamW_layers/step_5000/mlp.c_fc/full_hessian_block5.png
  [saved] hessians_adamW_layers/step_5000/mlp.c_fc/inv_hessian_full_block5.png
  [saved] hessians_adamW_layers/step_5000/mlp.c_fc/full_fisher_block5.png
  [saved] hessians_adamW_layers/step_5000/mlp.c_fc/inv_fisher_full_block5.png
  Hessian block5/mlp.c

Hessian step=5000: 100%|██████████| 6/6 [21:19<00:00, 213.23s/it]

  [saved] hessians_adamW_layers/step_5000/mlp.c_proj/inv_fisher_full_block5.png



Done!
